# Experiment 4

# Required libraries

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from pathlib import Path
import os, re, json
import pandas as pd
import timm
import numpy as np
from timm.utils import AttentionExtract
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, roc_auc_score,
    average_precision_score, precision_recall_fscore_support,
    confusion_matrix, classification_report
)
from torchvision.models.feature_extraction import create_feature_extractor, get_graph_node_names
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import *
from collections import defaultdict, Counter
import seaborn as sns
from torchvision import datasets, transforms
from tqdm import tqdm
from scipy.signal import medfilt
from scipy.ndimage import gaussian_filter1d
import torch.optim as optim
import warnings
warnings.filterwarnings('ignore')

import pickle
from itertools import groupby
import random
from sklearn.metrics import precision_recall_curve, roc_curve, auc


# Some sequence information

## Regex check 

In [2]:

pattern = re.compile(
    r'(?P<station>.+?)_'
    r'(?P<image_id>\d+_[0-9\.]+)_patches_patch_'
    r'(?P<x>\d+)_'
    r'(?P<y>\d+)_'
    r'(?P<angle>[0-9\.]+)_'
    r'(?P<track>-?\d+)'
    r'(?:_(?P<side>left|right))?'
    r'(?:\.[^.]+)?$'
)

# Example filenames 
examples = [
    "9_station_ruebenkamp_9.7_074_1631719553.500000028_patches_patch_1572_2199_107.070_0_left",
    "1_calibration_1.2_004_1631441714.500000016_patches_patch_1543_2199_108.037_0_left",
    "18_vegetation_switch_18.1_081_1631625169.200000016_patches_patch_2081_2199_80.458_0_right",
]

print("Testing the regex\n")

for f in examples:
    m = pattern.match(f)
    if m:
        print(f"MATCH: {f}")
        for k, v in m.groupdict().items():
            print(f"  {k:10s} = {v}")
    else:
        print(f"NO MATCH: {f}")
    print("-"*10)


Testing the regex

MATCH: 9_station_ruebenkamp_9.7_074_1631719553.500000028_patches_patch_1572_2199_107.070_0_left
  station    = 9_station_ruebenkamp_9.7
  image_id   = 074_1631719553.500000028
  x          = 1572
  y          = 2199
  angle      = 107.070
  track      = 0
  side       = left
----------
MATCH: 1_calibration_1.2_004_1631441714.500000016_patches_patch_1543_2199_108.037_0_left
  station    = 1_calibration_1.2
  image_id   = 004_1631441714.500000016
  x          = 1543
  y          = 2199
  angle      = 108.037
  track      = 0
  side       = left
----------
MATCH: 18_vegetation_switch_18.1_081_1631625169.200000016_patches_patch_2081_2199_80.458_0_right
  station    = 18_vegetation_switch_18.1
  image_id   = 081_1631625169.200000016
  x          = 2081
  y          = 2199
  angle      = 80.458
  track      = 0
  side       = right
----------


## check how many good and how many defective patches per sequence

In [ ]:
TRAIN_ROOT = Path("./blurclahe_data_folds/blurclahe_inner_seed79/folds/fold_2/train")
VAL_ROOT = Path("./blurclahe_data_folds/blurclahe_inner_seed79/folds/fold_2/val")

pattern = re.compile(
    r'(?P<station>.+?)_'
    r'(?P<image_id>\d+_[0-9\.]+)_patches_patch_'
    r'(?P<x>\d+)_'
    r'(?P<y>\d+)_'
    r'(?P<angle>[0-9\.]+)_'
    r'(?P<track>-?\d+)'
    r'(?:_(?P<side>left|right))?'
    r'(?:\.[^.]+)?$'
)

def get_seq_id_from_filename(fn: str):
    m = pattern.search(os.path.basename(fn))
    if not m:
        return None
    d = m.groupdict()
    seq_id = f"{d['station']}_{d['track']}_{d.get('side','')}"
    return seq_id

def count_patches_by_seq(root_dir: Path):
    """
    Input:  directory with subfolders "defective" and "good",
    Goal: count patches per sequence, returning a DataFrame with columns:
      seq_id, n_defective, n_good, total
    """
    records = []
    for cls in ["defective", "good"]:
        cls_dir = root_dir / cls
        if not cls_dir.exists():
            continue
        for fn in os.listdir(cls_dir):
            if not fn.lower().endswith((".png", ".jpg", ".jpeg", ".bmp")):
                continue
            full = cls_dir / fn
            seq_id = get_seq_id_from_filename(fn)
            if seq_id is None:
                continue
            records.append({"seq_id": seq_id, "class": cls})
    df = pd.DataFrame(records)
    # group
    counts = df.pivot_table(index="seq_id", columns="class", aggfunc="size", fill_value=0)
    # ensure columns exist
    if "defective" not in counts.columns:
        counts["defective"] = 0
    if "good" not in counts.columns:
        counts["good"] = 0
    counts = counts.reset_index().rename_axis(None, axis=1)
    counts["total"] = counts["defective"] + counts["good"]
    return counts

# Both for train and val
print("Train set sequences patch counts:")
train_counts = count_patches_by_seq(TRAIN_ROOT)
print(train_counts)

print("\nVal set sequences patch counts:")
val_counts = count_patches_by_seq(VAL_ROOT)
print(val_counts)


Train set sequences patch counts:
                                        seq_id  defective  good  total
0             10_station_suelldorf_10.1_0_left          0    10     10
1             13_station_ohlsdorf_13.1_-1_left          0    10     10
2            13_station_ohlsdorf_13.1_-1_right          0    10     10
3             13_station_ohlsdorf_13.1_0_right          1     9     10
4              14_signals_station_14.1_-1_left          0    10     10
5               14_signals_station_14.1_0_left          1     9     10
6              14_signals_station_14.1_0_right          1     9     10
7              14_signals_station_14.3_0_right          0    10     10
8          15_construction_vehicle_15.1_0_left          0    10     10
9         15_construction_vehicle_15.1_0_right          1     9     10
10               17_signal_bridge_17.1_0_right          0    10     10
11               17_signal_bridge_17.1_1_right          0    10     10
12            18_vegetation_switch_18.1_0_l

## statistics

In [4]:
pattern = re.compile(
    r'(?P<station>.+?)_'
    r'(?P<image_id>\d+_[0-9\.]+)_patches_patch_'
    r'(?P<x>\d+)_'
    r'(?P<y>\d+)_'
    r'(?P<angle>[0-9\.]+)_'
    r'(?P<track>-?\d+)'
    r'(?:_(?P<side>left|right))?'
    r'(?:\.[^.]+)?$'
)

def parse_filename(filename):
    """Extract metadata from filename."""
    match = pattern.match(filename)
    if match:
        return match.groupdict()
    return None

def analyze_sequences(fold_path):
    """Analyze sequential structure of the dataset."""
    fold_path = Path(fold_path)
    
    results = {
        'train': {'defective': [], 'good': []},
        'val': {'defective': [], 'good': []}
    }
    
    for split in ['train', 'val']:
        for label in ['defective', 'good']:
            img_dir = fold_path / split / label
            if not img_dir.exists():
                continue
                
            for img_path in img_dir.iterdir():
                if img_path.is_file() and img_path.suffix in ['.png', '.jpg', '.jpeg']:
                    parsed = parse_filename(img_path.name)
                    if parsed:
                        parsed['label'] = label
                        parsed['split'] = split
                        parsed['filename'] = img_path.name
                        results[split][label].append(parsed)
    
    return results

def compute_sequence_stats(data_dict):
    """Compute statistics about sequences."""
    all_data = []
    for split in ['train', 'val']:
        for label in ['defective', 'good']:
            all_data.extend(data_dict[split][label])
    
    df = pd.DataFrame(all_data)
    
    # Create sequence ID
    df['sequence_id'] = df['station'] + '_' + df['track'].astype(str) + '_' + df['side'].fillna('none')
    
    # Sort by sequence and image_id (timestamp)
    df = df.sort_values(['sequence_id', 'image_id'])
    
    print("-"*10)
    print("DATASET SEQUENTIAL STRUCTURE ANALYSIS")
    print("-"*10)
    
    print("\n1. OVERALL STATISTICS:")
    print(f"   Total patches: {len(df)}")
    print(f"   Train patches: {len(df[df['split']=='train'])}")
    print(f"   Val patches: {len(df[df['split']=='val'])}")
    print(f"   Defective patches: {len(df[df['label']=='defective'])}")
    print(f"   Good patches: {len(df[df['label']=='good'])}")
    
    print("\n2. SEQUENCE STATISTICS:")
    print(f"   Total unique sequences: {df['sequence_id'].nunique()}")
    
    # Sequences per split
    train_seqs = set(df[df['split']=='train']['sequence_id'].unique())
    val_seqs = set(df[df['split']=='val']['sequence_id'].unique())
    print(f"   Train sequences: {len(train_seqs)}")
    print(f"   Val sequences: {len(val_seqs)}")
    print(f"   Overlapping sequences (SHOULD BE 0): {len(train_seqs & val_seqs)}")
    
    # Sequence lengths
    seq_lengths = df.groupby('sequence_id').size()
    print(f"\n3. SEQUENCE LENGTH STATISTICS:")
    print(f"   Min sequence length: {seq_lengths.min()}")
    print(f"   Max sequence length: {seq_lengths.max()}")
    print(f"   Mean sequence length: {seq_lengths.mean():.2f}")
    print(f"   Median sequence length: {seq_lengths.median():.0f}")
    
    # Label distribution within sequences
    print(f"\n4. LABEL PATTERNS IN SEQUENCES:")
    seq_label_counts = df.groupby(['sequence_id', 'label']).size().unstack(fill_value=0)
    
    pure_defective = (seq_label_counts['defective'] > 0) & (seq_label_counts['good'] == 0)
    pure_good = (seq_label_counts['good'] > 0) & (seq_label_counts['defective'] == 0)
    mixed = (seq_label_counts['defective'] > 0) & (seq_label_counts['good'] > 0)
    
    print(f"   Pure defective sequences: {pure_defective.sum()}")
    print(f"   Pure good sequences: {pure_good.sum()}")
    print(f"   Mixed sequences (both labels): {mixed.sum()}")
    
    # Defect concentration in mixed sequences
    if mixed.sum() > 0:
        mixed_seqs = seq_label_counts[mixed]
        defect_ratio = mixed_seqs['defective'] / (mixed_seqs['defective'] + mixed_seqs['good'])
        print(f"   Avg defect ratio in mixed sequences: {defect_ratio.mean():.3f}")
    
    print("\n5. TEMPORAL PATTERNS:")
    # Check if defects cluster together
    defect_runs = []
    for seq_id in df['sequence_id'].unique():
        seq_df = df[df['sequence_id'] == seq_id].sort_values('image_id')
        labels = (seq_df['label'] == 'defective').astype(int).values
        
        # Count runs of consecutive defects
        if len(labels) > 1:
            runs = []
            current_run = 1
            for i in range(1, len(labels)):
                if labels[i] == labels[i-1] == 1:
                    current_run += 1
                else:
                    if labels[i-1] == 1:
                        runs.append(current_run)
                    current_run = 1
            if labels[-1] == 1:
                runs.append(current_run)
            defect_runs.extend(runs)
    
    if defect_runs:
        print(f"   Defect runs found: {len(defect_runs)}")
        print(f"   Avg consecutive defect length: {np.mean(defect_runs):.2f}")
        print(f"   Max consecutive defect length: {max(defect_runs)}")
    
    return df, seq_label_counts

# Run analysis
fold2_path = "./blurclahe_data_folds/blurclahe_inner_seed79/folds/fold_2"
data = analyze_sequences(fold2_path)
df, seq_stats = compute_sequence_stats(data)

# Save for later use
df.to_csv("fold2_sequence_analysis.csv", index=False)
print("\n Saved sequence analysis to 'fold2_sequence_analysis.csv'")

----------
DATASET SEQUENTIAL STRUCTURE ANALYSIS
----------

1. OVERALL STATISTICS:
   Total patches: 1300
   Train patches: 1080
   Val patches: 220
   Defective patches: 325
   Good patches: 975

2. SEQUENCE STATISTICS:
   Total unique sequences: 58
   Train sequences: 45
   Val sequences: 13
   Overlapping sequences (SHOULD BE 0): 0

3. SEQUENCE LENGTH STATISTICS:
   Min sequence length: 10
   Max sequence length: 100
   Mean sequence length: 22.41
   Median sequence length: 10

4. LABEL PATTERNS IN SEQUENCES:
   Pure defective sequences: 1
   Pure good sequences: 32
   Mixed sequences (both labels): 25
   Avg defect ratio in mixed sequences: 0.310

5. TEMPORAL PATTERNS:
   Defect runs found: 40
   Avg consecutive defect length: 8.12
   Max consecutive defect length: 100

 Saved sequence analysis to 'fold2_sequence_analysis.csv'


# creates the train_predictions of the model to be used for training the lstm (in case of not saved by original run)

In [ ]:

def generate_predictions_with_filenames(
    fold_root: Path,
    data_dir: Path,
    split: str = "train"  # "train" or "val"
):
    """
    Generate train predictions CSV with filenames included.
    Args:
        fold_root: Path to fold directory (e.g., resultsexp3/.../fold_4/seed0)
        data_dir: Path to data directory (e.g., blurclahe_data_folds/.../fold_4/train)
        split: "train" or "val"
    """
    print(f"\n{'-'*10}")
    print(f"Generating {split}_predictions.csv with filenames")
    print(f"{'-'*10}")
    
    # Load checkpoint and config
    ckpt_path = fold_root / "artifacts" / "best_model.pt"
    mapping = json.load(open(fold_root / "class_mapping.json"))
    class_names = mapping["class_names"]
    num_classes = len(class_names)
    
    # Load model
    print(f"Loading model from {ckpt_path}")
    model = timm.create_model("deit_small_patch16_224", pretrained=False, num_classes=num_classes)
    model.load_state_dict(torch.load(ckpt_path, map_location="cuda"))
    model.eval().to("cuda")
    # from torchvision import models
    # model = models.resnet18(weights=None)  # no pretrained weights
    # model.fc = torch.nn.Linear(model.fc.in_features, num_classes)  # replace classifier head
    # model.load_state_dict(torch.load(ckpt_path, map_location="cuda"))
    # model.eval().to("cuda")

    # Prepare data
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    
    ds = datasets.ImageFolder(data_dir, transform=transform)
    loader = torch.utils.data.DataLoader(ds, batch_size=64, shuffle=False, num_workers=0)
    
    # Extract filenames from dataset
    # ImageFolder stores samples as (path, class_idx)
    all_paths = [Path(path).name for path, _ in ds.samples]
    
    print(f"Total samples: {len(all_paths)}")
    print(f"Example filename: {all_paths[0]}")
    
    records = []
    idx = 0
    
    with torch.no_grad():
        for imgs, labels in tqdm(loader, desc=f"Inference on {split}"):
            imgs = imgs.cuda()
            labels = labels.cuda()
            
            logits = model(imgs)
            probs = torch.softmax(logits, dim=1).cpu().numpy()
            preds = probs.argmax(axis=1)
            
            batch_size = len(labels)
            for i in range(batch_size):
                row = {
                    "filename": all_paths[idx],
                    "y_true": int(labels[i].cpu()),
                    "y_pred": int(preds[i])
                }
                for c in range(num_classes):
                    row[f"p_class_{c}"] = float(probs[i, c])
                records.append(row)
                idx += 1
    
    # Save
    df = pd.DataFrame(records)
    output_path = fold_root / f"{split}_predictions.csv"
    df.to_csv(output_path, index=False)
    
    print(f"Saved {output_path}")
    print(f"   Total predictions: {len(df)}")
    print(f"   Columns: {list(df.columns)}")
    print(f"\nFirst few rows:")
    print(df.head())
    
    return df


if __name__ == "__main__":
    
    base_test_dir = Path("results_experiment3/DeiT-Small16/blurclahe_inner_seed79")
    base_data_dir = Path("blurclahe_data_folds/blurclahe_inner_seed79/folds")

    # base_test_dir = Path("blurclahe_inner_seed0")
    # base_data_dir = Path("blurclahe_data_folds/blurclahe_inner_seed0/folds")

    seeds = [0, 42, 123]
    folds = range(5)  # fold_0 .. fold_4

    for fold in folds:
        for seed in seeds:
            fold_root = base_test_dir / f"fold_{fold}" / f"seed{seed}"
            data_root = base_data_dir / f"fold_{fold}" / "train"

            print(f"Generating predictions for fold {fold}, seed {seed}...")
            generate_predictions_with_filenames(
                fold_root=fold_root,
                data_dir=data_root,
                split="train" #or val
            )
    
    print("\n" + "-"*10)
    print("All predictions regenerated with filenames!")
    print("-"*10)

# Post processing methods to be applied

In [ ]:
EXP3_ROOT = Path("./results_experiment3/DeiT-Small16/blurclahe_inner_seed79") #results from exp3 folder with folds 
FOLDS_ROOT = Path("blurclahe_data_folds/blurclahe_inner_seed79/folds") #original data folds
OUTPUT_DIR = Path("exp4_results")
OUTPUT_DIR.mkdir(exist_ok=True)
RANDOM_SEED = 42

# Search grids
SIGMA_GRID = [0.5, 1.0, 1.5, 2.0]
CONF_THRESH_GRID = [0.05, 0.1, 0.2]

MULTISCALE_SIGMA_SETS = [
    [0.5, 1.0, 1.5, 2.0],
    [1.0, 2.0, 3.0],
    [0.5, 1.0, 2.0, 3.0, 4.0],
    [0.5,1,2], 
    [1,2,4]
    
]

FILENAME_PATTERN = re.compile(
    r'(?P<station>.+?)_'
    r'(?P<image_id>\d+_[0-9\.]+)_patches_patch_'
    r'(?P<x>\d+)_'
    r'(?P<y>\d+)_'
    r'(?P<angle>[0-9\.]+)_'
    r'(?P<track>-?\d+)'
    r'(?:_(?P<side>left|right))?'
    r'(?:\.[^.]+)?$'
)




# LSTM FUNCTIONS START

# Model
class TemporalRefinementLSTM(nn.Module):
    """LSTM-based temporal refinement of defect probabilities."""
    def __init__(self, input_size=3, hidden_size=64, num_layers=2, dropout=0.3):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=True
        )
        
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size * 2, 2)
        
    def forward(self, x, lengths=None):
        if lengths is not None:
            x_packed = nn.utils.rnn.pack_padded_sequence(
                x, lengths.cpu(), batch_first=True, enforce_sorted=False
            )
            lstm_out, _ = self.lstm(x_packed)
            lstm_out, _ = nn.utils.rnn.pad_packed_sequence(lstm_out, batch_first=True)
        else:
            lstm_out, _ = self.lstm(x)
        
        lstm_out = self.dropout(lstm_out)
        logits = self.fc(lstm_out)
        return logits


# Dataset 
class SequenceDataset(Dataset):
    def __init__(self, sequences, max_len=None):
        self.sequences = sequences
        self.max_len = max_len or max(len(s[0]) for s in sequences)
    
    def __len__(self):
        return len(self.sequences)
    
    def __getitem__(self, idx):
        features, labels, metadata = self.sequences[idx]
        seq_len = len(features)
        
        if seq_len < self.max_len:
            pad_len = self.max_len - seq_len
            features = np.vstack([features, np.zeros((pad_len, features.shape[1]))])
            labels = np.concatenate([labels, np.zeros(pad_len, dtype=labels.dtype)])
        
        return {
            'features': torch.FloatTensor(features),
            'labels': torch.LongTensor(labels),
            'length': torch.LongTensor([seq_len]),
            'metadata': metadata
        }


def collate_fn(batch):
    return {
        'features': torch.stack([item['features'] for item in batch]),
        'labels': torch.stack([item['labels'] for item in batch]),
        'length': torch.cat([item['length'] for item in batch]),
        'metadata': [item['metadata'] for item in batch]
    }



# Data Preparation
def prepare_sequences_for_lstm(df_per_seed, build_sequences_fn):
    """Convert ensemble predictions into LSTM-ready sequences."""
    df_per_seed = align_seed_frames(df_per_seed)
    probs_all = np.array([df_per_seed[s]["p_defective"].values for s in df_per_seed])
    mean_probs = np.mean(probs_all, axis=0)
    std_probs = np.std(probs_all, axis=0)
    
    df_base = list(df_per_seed.values())[0].copy()
    df_base["p_defective"] = mean_probs
    df_base["p_good"] = 1 - mean_probs
    df_base["std_seeds"] = std_probs
    
    sequences_raw = build_sequences_fn(df_base)
    
    sequences = []
    for probs, labels, idxs in sequences_raw:
        seq_df = df_base.loc[idxs]
        
        features = np.column_stack([
            seq_df["p_defective"].values,
            seq_df["p_good"].values,
            seq_df["std_seeds"].values
        ])
        
        labels = seq_df["y_true_defective"].values.astype(np.int64)
        
        metadata = {
            'length': len(features),
            'station': seq_df.iloc[0].get('station', 'unknown'),
            'track': seq_df.iloc[0].get('track', 'unknown'),
            'side':  seq_df.iloc[0].get('side', 'unknown'),   
            'indices': idxs.tolist()
        }

        
        sequences.append((features, labels, metadata))
    
    return sequences


# Training 
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    all_preds, all_labels = [], []
    
    for batch in loader:
        features = batch['features'].to(device)
        labels = batch['labels'].to(device)
        lengths = batch['length'].to(device)
        
        optimizer.zero_grad()
        logits = model(features, lengths)
        
        # Loss only on non-padded positions
        loss = 0
        for i, length in enumerate(lengths):
            loss += criterion(logits[i, :length], labels[i, :length])
        loss = loss / len(lengths)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        total_loss += loss.item()
        
        probs = torch.softmax(logits, dim=-1)
        for i, length in enumerate(lengths):
            all_preds.extend(probs[i, :length, 1].detach().cpu().numpy())
            all_labels.extend(labels[i, :length].cpu().numpy())
    
    return total_loss / len(loader), np.array(all_preds), np.array(all_labels)


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds, all_labels, all_indices = [], [], []
    
    for batch in loader:
        features = batch['features'].to(device)
        labels = batch['labels'].to(device)
        lengths = batch['length'].to(device)
        
        logits = model(features, lengths)
        
        loss = 0
        for i, length in enumerate(lengths):
            loss += criterion(logits[i, :length], labels[i, :length])
        loss = loss / len(lengths)
        total_loss += loss.item()
        
        probs = torch.softmax(logits, dim=-1)
        for i, length in enumerate(lengths):
            all_preds.extend(probs[i, :length, 1].cpu().numpy())
            all_labels.extend(labels[i, :length].cpu().numpy())
            all_indices.extend(batch['metadata'][i]['indices'])
    
    return total_loss / len(loader), np.array(all_preds), np.array(all_labels), all_indices


def compute_metrics_lstm(y_true, y_pred_probs):
    y_pred = (y_pred_probs >= 0.5).astype(int)
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, pos_label=1, average='binary', zero_division=0
    )
    try:
        auprc = average_precision_score(y_true, y_pred_probs)
    except:
        auprc = np.nan
    return {'precision': float(prec), 'recall': float(rec), 'f1': float(f1), 'auprc': float(auprc)}


#  Training with Inner Cross-Validation
def train_lstm_with_inner_cv(sequences, hidden_size, num_layers, dropout, lr, 
                              epochs=50, batch_size=16, device='cuda', 
                              patience=10, val_split=0.2, random_state=42):
    """
    Train LSTM with inner train/val split for early stopping.
    Important: Ensures sequences from same (station, track, side) are NOT split
    across train_inner and val_inner to prevent temporal leakage.
    Args:
        sequences: All training sequences (from train set of fold)
        hidden_size, num_layers, dropout, lr: Hyperparameters
        val_split: Fraction of sequences to use for inner validation
    Returns:
        model: Trained model (on train_inner, early stopped on val_inner)
        val_inner_metrics: Metrics on inner validation set
    """

    # SEQUENCE-AWARE SPLITTING (prevents temporal leakage)
    # Group sequences by (station, track, side) to keep segments together
    
    
    sequence_groups = defaultdict(list)
    for idx, (features, labels, metadata) in enumerate(sequences):
        # Create unique key for this track segment
        # Convert values to strings to ensure hashability
        station = str(metadata.get('station', 'unknown'))
        track = str(metadata.get('track', 'unknown'))
        side = metadata.get('side', 'unknown')
        # Handle numpy arrays or other non-hashable types
        if isinstance(side, np.ndarray):
            side = str(side)
        elif side is None:
            side = 'unknown'
        else:
            side = str(side)
        key = (station, track, side)
        sequence_groups[key].append(idx)
    
    # Split at the GROUP level (not individual sequences)
    group_keys = list(sequence_groups.keys())
    n_groups = len(group_keys)
    n_val_groups = max(1, int(n_groups * val_split))
    
    # Shuffle groups with fixed seed
    np.random.seed(random_state)
    shuffled_indices = np.random.permutation(n_groups)
    val_keys = [group_keys[i] for i in shuffled_indices[:n_val_groups]]
    train_keys = [group_keys[i] for i in shuffled_indices[n_val_groups:]]
    
    # Collect sequence indices
    train_indices = []
    for key in train_keys:
        train_indices.extend(sequence_groups[key])
    
    val_indices = []
    for key in val_keys:
        val_indices.extend(sequence_groups[key])
    
    train_sequences = [sequences[i] for i in train_indices]
    val_sequences = [sequences[i] for i in val_indices]
    
    # Verify no overlap (sanity check)
    train_tracks = set((s[2]['station'], s[2]['track'], s[2].get('side')) for s in train_sequences)
    val_tracks = set((s[2]['station'], s[2]['track'], s[2].get('side')) for s in val_sequences)
    assert len(train_tracks & val_tracks) == 0, "ERROR: Track leakage between train/val!"
    
    print(f"    Inner split: {len(train_sequences)} train seqs ({len(train_keys)} tracks), "
          f"{len(val_sequences)} val seqs ({len(val_keys)} tracks)")
    
    # Create datasets
    train_dataset = SequenceDataset(train_sequences)
    val_dataset = SequenceDataset(val_sequences, max_len=train_dataset.max_len)
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, 
                             shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, 
                           shuffle=False, collate_fn=collate_fn)
    
    # Compute class weights
    all_labels = np.concatenate([seq[1] for seq in train_sequences])
    class_counts = np.bincount(all_labels, minlength=2)
    class_counts = np.clip(class_counts, 1, None)  #prevent zero division
    class_weights = torch.FloatTensor(class_counts.max() / class_counts).to(device)


    # Model
    model = TemporalRefinementLSTM(
        input_size=3, hidden_size=hidden_size, 
        num_layers=num_layers, dropout=dropout
    ).to(device)
    
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    
    # Training loop with early stopping
    best_val_auprc = -np.inf
    best_model_state = None
    patience_counter = 0
    
    for epoch in range(epochs):
        train_loss, train_probs, train_labels = train_one_epoch(
            model, train_loader, optimizer, criterion, device
        )
        val_loss, val_probs, val_labels, _ = evaluate(
            model, val_loader, criterion, device
        )
        
        val_metrics = compute_metrics_lstm(val_labels, val_probs)
        
        # Early stopping
        if val_metrics['auprc'] > best_val_auprc:
            best_val_auprc = val_metrics['auprc']
            best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break
        print(f"Epoch= {epoch}, "f"train loss={train_loss:.4f}," f"AUPRC={val_metrics['auprc']:.4f}, "
                          f"F1={val_metrics['f1']:.4f}, "
                          f"Prec={val_metrics['precision']:.4f}, "
                          f"Rec={val_metrics['recall']:.4f}")
        
    
    # Load best model
    model.load_state_dict(best_model_state)
    
    # Return model trained on train_inner, evaluated on val_inner
    return model, val_metrics


def retrain_on_full_train(sequences, hidden_size, num_layers, dropout, lr,
                           epochs=50, batch_size=16, device='cuda'):
    """
    Retrain model on ALL training sequences with fixed hyperparameters.
    Saves best model based on training loss (since no validation set available).
    """
    dataset = SequenceDataset(sequences)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
    
    # Compute class weights
    all_labels = np.concatenate([seq[1] for seq in sequences])
    class_counts = np.bincount(all_labels, minlength=2)
    class_counts = np.clip(class_counts, 1, None)  # prevent zero division
    class_weights = torch.FloatTensor(class_counts.max() / class_counts).to(device)

    model = TemporalRefinementLSTM(
        input_size=3, hidden_size=hidden_size,
        num_layers=num_layers, dropout=dropout
    ).to(device)
    
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    
    # Track best model by training auprc
    print(f"    Training for up to {epochs} epochs ")
    best_train_auprc = -np.inf
    best_model_state = None
    best_epoch = -1

    for epoch in range(epochs):
        train_loss, train_probs, train_labels = train_one_epoch(
            model, loader, optimizer, criterion, device
        )

        # Compute training AUPRC 
        train_metrics = compute_metrics_lstm(train_labels, train_probs)
        train_auprc = train_metrics['auprc']
        # Track best epoch
        if train_auprc > best_train_auprc:
            best_train_auprc = train_auprc
            best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            best_epoch = epoch
        
        if (epoch + 1) % 10 == 0:
            print(f"    Epoch {epoch+1:3d}/{epochs} | AUPRC: {train_auprc:.4f}")
    
    model.load_state_dict(best_model_state)
    print(f"    Using model from epoch {best_epoch+1} (AUPRC: {best_train_auprc:.4f})")
    return model



# Hyperparameter Tuning 

def tune_lstm(train_sequences, device='cuda'):
    """
    Tune LSTM hyperparameters on train set using inner cross-validation
    Process:
      1. For each hyperparameter combination:
         - Split train into train_inner/val_inner (80/20) BY TRACK GROUP
         - Train on train_inner with early stopping on val_inner
         - Record val_inner performance
      2. Select best hyperparameters based on val_inner AUPRC
      3. Return best hyperparameters (NOT the model yet)
      - No sequence leakage (tracks kept together in splits)
      - Same selection criterion as other methods (precision with recall constraint)
      - Hyperparameters selected on train, evaluated on held-out val 
    Args:
        train_sequences: Training sequences from fold
        
    Returns:
        best_params: Dict with best hyperparameters
        best_metrics: Validation metrics for best params
    """
    print("\n  Grid search for LSTM hyperparameters:")
    print(" (Using sequence-aware inner CV to prevent temporal leakage)")
    
    # Search grid 
    param_grid = {
        'hidden_size': [32, 64],
        'num_layers': [1, 2],
        'dropout': [0.2, 0.3],
        'lr': [1e-3, 5e-4]
    }
    
    # verify we have enough track groups for splitting
   
    sequence_groups = defaultdict(list)
    for idx, (features, labels, metadata) in enumerate(train_sequences):
        # Create unique key for this track segment
        # Convert values to strings to ensure hashability
        station = str(metadata.get('station', 'unknown'))
        track = str(metadata.get('track', 'unknown'))
        side = metadata.get('side', 'unknown')
        # Handle numpy arrays or other non-hashable types
        if isinstance(side, np.ndarray):
            side = str(side)
        elif side is None:
            side = 'unknown'
        else:
            side = str(side)
        key = (station, track, side)
        sequence_groups[key].append(idx)

    
    n_track_groups = len(sequence_groups)
    print(f"     Total sequences: {len(train_sequences)}")
    print(f"     Unique track groups: {n_track_groups}")
    
    if n_track_groups < 5:
        print(f"     WARNING: Only {n_track_groups} track groups available!")
        print(f"     Recommend using at least 5 groups for reliable inner CV.")
    
    candidates = []
    
    for hidden_size in param_grid['hidden_size']:
        for num_layers in param_grid['num_layers']:
            for dropout in param_grid['dropout']:
                for lr in param_grid['lr']:
                    print(f"    hidden={hidden_size}, layers={num_layers}, "
                          f"dropout={dropout}, lr={lr}", end=" → ")
                    
                    # Train with inner CV
                    model, val_metrics = train_lstm_with_inner_cv(
                        train_sequences,
                        hidden_size=hidden_size,
                        num_layers=num_layers,
                        dropout=dropout,
                        lr=lr,
                        epochs=100,
                        batch_size=16,
                        device=device,
                        patience=30,
                        val_split=0.2
                    )
                    
                    print(f"AUPRC={val_metrics['auprc']:.4f}, "
                          f"F1={val_metrics['f1']:.4f}, "
                          f"Prec={val_metrics['precision']:.4f}, "
                          f"Rec={val_metrics['recall']:.4f}")
                    
                    params = {
                        'hidden_size': hidden_size,
                        'num_layers': num_layers,
                        'dropout': dropout,
                        'lr': lr
                    }
                    candidates.append((params, val_metrics))
    
    # Select best based on criterion (from select_best_model)
    valid = [(p, m) for p, m in candidates if m["recall"] >= 0.75]
    if valid:
        best = max(valid, key=lambda x: x[1]["precision"])
    else:
        best = max(candidates, key=lambda x: x[1]["f1"])
    
    best_params, best_metrics = best
    
    print(f"  Best LSTM params: {best_params}")
    print(f"  Prec={best_metrics['precision']:.4f}, "
          f"Rec={best_metrics['recall']:.4f}, "
          f"F1={best_metrics['f1']:.4f}, "
          f"AUPRC={best_metrics['auprc']:.4f}")
    
    return best_params, best_metrics


# Final Evaluation 

def lstm_temporal_refinement(train_df_per_seed, val_df_per_seed, build_sequences_fn,
                             best_params, device='cuda'):
    """
    Apply LSTM refinement using tuned hyperparameters.   
    Process:
      1. Prepare train sequences and retrain model on FULL train set with best_params
      2. Apply trained model to val set
      3. Return metrics on val set
    Args:
        train_df_per_seed: Training data (to retrain final model)
        val_df_per_seed: Validation data (to evaluate)
        build_sequences_fn: sequence building function
        best_params: Hyperparameters from tune_lstm()       
    Returns:
        metrics: Dict with val set performance
    """
    print("\n  → Retraining LSTM on full train set with best params...")
    
    # Prepare sequences
    train_sequences = prepare_sequences_for_lstm(train_df_per_seed, build_sequences_fn)
    val_sequences = prepare_sequences_for_lstm(val_df_per_seed, build_sequences_fn)
    
    # Retrain on full train set (no validation split, fixed hyperparams)
    model = retrain_on_full_train(
        train_sequences,
        hidden_size=best_params['hidden_size'],
        num_layers=best_params['num_layers'],
        dropout=best_params['dropout'],
        lr=best_params['lr'],
        epochs=100,  
        batch_size=16,
        device=device
    )
    
    print("-> Evaluating on validation set...")
    
    # Evaluate on val set
    val_dataset = SequenceDataset(val_sequences, max_len=max(len(s[0]) for s in train_sequences))
    val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, collate_fn=collate_fn)
    
    model.eval()
    all_probs = []
    all_labels = []
    
    with torch.no_grad():
        for batch in val_loader:
            features = batch['features'].to(device)
            lengths = batch['length'].to(device)
            labels = batch['labels']
            
            logits = model(features, lengths)
            probs = torch.softmax(logits, dim=-1)
            
            for i, length in enumerate(lengths):
                all_probs.extend(probs[i, :length, 1].cpu().numpy())
                all_labels.extend(labels[i, :length].numpy())
    
    all_probs = np.array(all_probs)
    all_labels = np.array(all_labels)
    all_preds = (all_probs >= 0.5).astype(int)
    
    # Compute metrics
    
    prec, rec, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, pos_label=1, average='binary', zero_division=0
    )
    
    metrics = {
        'accuracy': float(accuracy_score(all_labels, all_preds)),
        'balanced_accuracy': float(balanced_accuracy_score(all_labels, all_preds)),
        'precision': float(prec),
        'recall': float(rec),
        'f1': float(f1)
    }
    
    try:
        metrics['auprc'] = float(average_precision_score(all_labels, all_probs))
        metrics['auroc'] = float(roc_auc_score(all_labels, all_probs))  # placeholder
    except:
        metrics['auprc'] = np.nan
        metrics['auroc'] = np.nan
    
    cm = confusion_matrix(all_labels, all_preds, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    metrics.update({'tp': int(tp), 'tn': int(tn), 'fp': int(fp), 'fn': int(fn)})
    
    return metrics, model


def verify_experimental_fairness(train_df_per_seed, val_df_per_seed, 
                                  build_sequences, OUTPUT_DIR):
    """
    Verify that LSTM soed inner CV fairly
    """
    
    print("EXPERIMENTAL FAIRNESS VERIFICATION")
    
    
    report = []
    
    # 1. Data splits
    train_df = list(train_df_per_seed.values())[0]
    val_df = list(val_df_per_seed.values())[0]
    
    train_seqs = build_sequences(train_df.copy())
    val_seqs = build_sequences(val_df.copy())
    
    # Extract track identifiers
    def get_track_ids(sequences, df):
        track_ids = set()
        for _, _, idxs in sequences:
            sample_row = df.iloc[idxs[0]]
            track_id = (sample_row.get('station', 'unknown'),
                       sample_row.get('track', 'unknown'),
                       sample_row.get('side', 'unknown'))
            track_ids.add(track_id)
        return track_ids
    
    train_tracks = get_track_ids(train_seqs, train_df)
    val_tracks = get_track_ids(val_seqs, val_df)
    
    # find tracks without side info (should not exist)
    problematic_tracks = [
        t for t in train_tracks.union(val_tracks)
        if not (isinstance(t, tuple) and len(t) == 3 and str(t[2]) in {"left", "right"})
    ]

    if problematic_tracks:
        print("Found tracks missing side info:")
        for t in problematic_tracks[:10]:
            print("   ", t)
        print(f"Total problematic tracks: {len(problematic_tracks)}")

        # locate corresponding filenames for debugging
        first_key = next(iter(train_df_per_seed))
        missing_df = train_df_per_seed[first_key].copy() 
        bad_matches = missing_df[
            missing_df["track"].astype(str).apply(lambda tr: any(str(tr) in str(pt) for pt in problematic_tracks))
        ]
        print("\nExample filenames possibly linked to missing-side tracks:")
        print(bad_matches["filename"].head())

    # Sanity check: each track ID should encode side information
    assert all(
        (isinstance(t, tuple) and len(t) == 3 and t[2] in ["left", "right"])
        or ("_left" in str(t) or "_right" in str(t))
        for t in train_tracks.union(val_tracks)
    ), "Missing side info in track IDs — check build_sequences() parsing or metadata."

    # Check: No overlap between train and val tracks
    overlap = train_tracks & val_tracks
    check1_pass = len(overlap) == 0
    
    report.append("-"*10)
    report.append("Train/Val Track Separation")
    report.append("-"*10)
    report.append(f"Train unique tracks: {len(train_tracks)}")
    report.append(f"Val unique tracks: {len(val_tracks)}")
    report.append(f"Overlapping tracks: {len(overlap)}")
    report.append(f"Status: {'PASS' if check1_pass else 'FAIL - DATA LEAKAGE!'}")
    
    if not check1_pass:
        report.append(f"Overlapping: {list(overlap)[:5]}...")
    
    return check1_pass



def visualize_lstm_fold_improvements(lstm_models_dict, folds_dict, output_dir, device='cuda'):
    """
    Comprehensive LSTM visualization across all folds.
    
    Args:
        lstm_models_dict: dict like {'fold_0': model_0, 'fold_1': model_1, ...}
        folds_dict: dict like {'fold_0': val_sequences_0, 'fold_1': val_sequences_1, ...}
        output_dir: where to save plots
        device: 'cuda' or 'cpu'
    """
    fold_metrics = []
    all_pr_curves = []
    all_roc_curves = []

    # Compute metrics per fold
    for fold_name in sorted(folds_dict.keys()):
        model = lstm_models_dict[fold_name]
        val_sequences = folds_dict[fold_name]
        
        model.eval()
        all_baseline, all_lstm, all_labels = [], [], []

        for features, labels, _ in val_sequences:
            baseline_probs = features[:, 0]

            with torch.no_grad():
                feat_tensor = torch.FloatTensor(features).unsqueeze(0).to(device)
                length = torch.LongTensor([len(features)]).to(device)
                logits = model(feat_tensor, length)
                lstm_probs = torch.softmax(logits, dim=-1)[0, :, 1].cpu().numpy()

            all_baseline.extend(baseline_probs)
            all_lstm.extend(lstm_probs)
            all_labels.extend(labels)

        # Convert to arrays
        all_baseline = np.array(all_baseline)
        all_lstm = np.array(all_lstm)
        all_labels = np.array(all_labels)

        # Compute metrics
        auprc_baseline = average_precision_score(all_labels, all_baseline)
        auprc_lstm = average_precision_score(all_labels, all_lstm)
        
        # Precision-Recall curves
        prec_base, rec_base, _ = precision_recall_curve(all_labels, all_baseline)
        prec_lstm, rec_lstm, _ = precision_recall_curve(all_labels, all_lstm)
        
        # ROC curves
        fpr_base, tpr_base, _ = roc_curve(all_labels, all_baseline)
        fpr_lstm, tpr_lstm, _ = roc_curve(all_labels, all_lstm)
        auroc_baseline = auc(fpr_base, tpr_base)
        auroc_lstm = auc(fpr_lstm, tpr_lstm)

        fold_metrics.append({
            'fold': fold_name,
            'baseline_AUPRC': auprc_baseline,
            'lstm_AUPRC': auprc_lstm,
            'delta_AUPRC': auprc_lstm - auprc_baseline,
            'baseline_AUROC': auroc_baseline,
            'lstm_AUROC': auroc_lstm,
            'delta_AUROC': auroc_lstm - auroc_baseline
        })
        
        all_pr_curves.append({
            'fold': fold_name,
            'baseline': (rec_base, prec_base),
            'lstm': (rec_lstm, prec_lstm)
        })
        
        all_roc_curves.append({
            'fold': fold_name,
            'baseline': (fpr_base, tpr_base),
            'lstm': (fpr_lstm, tpr_lstm)
        })

    df_metrics = pd.DataFrame(fold_metrics)
    print("LSTM PERFORMANCE BY FOLD")
    print(df_metrics.round(4).to_string(index=False))
    print("\nMean Improvement:")
    print(f"  Δ AUPRC: {df_metrics['delta_AUPRC'].mean():.4f} ± {df_metrics['delta_AUPRC'].std():.4f}")
    print(f"  Δ AUROC: {df_metrics['delta_AUROC'].mean():.4f} ± {df_metrics['delta_AUROC'].std():.4f}")

    # Create comprehensive figure with 4 subplots
    fig = plt.figure(figsize=(16, 12))
    gs = fig.add_gridspec(2, 2, hspace=0.3, wspace=0.3)

    # 1. AUPRC Improvement Bar Chart
    ax1 = fig.add_subplot(gs[0, 0])
    colors = ['darkgreen' if x > 0 else 'darkred' for x in df_metrics['delta_AUPRC']]
    bars = ax1.bar(df_metrics['fold'], df_metrics['delta_AUPRC'], 
                   color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
    ax1.axhline(0, color='black', linestyle='--', linewidth=1)
    ax1.set_ylabel('Δ AUPRC (LSTM - Baseline)', fontsize=11, fontweight='bold')
    ax1.set_xlabel('Fold', fontsize=11, fontweight='bold')
    ax1.set_title('AUPRC Improvement by Fold', fontsize=13, fontweight='bold')
    ax1.grid(axis='y', alpha=0.3)
    
    # Add value labels
    for bar, val in zip(bars, df_metrics['delta_AUPRC']):
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height,
                f'{val:+.4f}', ha='center', va='bottom' if val > 0 else 'top',
                fontsize=9, fontweight='bold')

    # 2. Baseline vs LSTM Comparison
    ax2 = fig.add_subplot(gs[0, 1])
    x = np.arange(len(df_metrics))
    width = 0.35
    ax2.bar(x - width/2, df_metrics['baseline_AUPRC'], width, 
            label='Baseline', alpha=0.7, color='coral', edgecolor='black')
    ax2.bar(x + width/2, df_metrics['lstm_AUPRC'], width,
            label='LSTM', alpha=0.7, color='steelblue', edgecolor='black')
    ax2.set_ylabel('AUPRC', fontsize=11, fontweight='bold')
    ax2.set_xlabel('Fold', fontsize=11, fontweight='bold')
    ax2.set_title('AUPRC: Baseline vs LSTM', fontsize=13, fontweight='bold')
    ax2.set_xticks(x)
    ax2.set_xticklabels(df_metrics['fold'])
    ax2.legend(fontsize=10)
    ax2.grid(axis='y', alpha=0.3)

    # 3. Precision-Recall Curves (all folds)
    ax3 = fig.add_subplot(gs[1, 0])
    colors_pr = plt.cm.tab10(np.linspace(0, 1, len(all_pr_curves)))
    
    for i, curve_data in enumerate(all_pr_curves):
        fold = curve_data['fold']
        rec_base, prec_base = curve_data['baseline']
        rec_lstm, prec_lstm = curve_data['lstm']
        
        # Baseline (dashed)
        ax3.plot(rec_base, prec_base, '--', color=colors_pr[i], 
                alpha=0.4, linewidth=1.5, label=f'{fold} Baseline')
        # LSTM (solid)
        ax3.plot(rec_lstm, prec_lstm, '-', color=colors_pr[i],
                alpha=0.8, linewidth=2, label=f'{fold} LSTM')
    
    ax3.set_xlabel('Recall', fontsize=11, fontweight='bold')
    ax3.set_ylabel('Precision', fontsize=11, fontweight='bold')
    ax3.set_title('Precision-Recall Curves (All Folds)', fontsize=13, fontweight='bold')
    ax3.legend(loc='best', fontsize=7, ncol=2)
    ax3.grid(alpha=0.3)
    ax3.set_xlim([0, 1])
    ax3.set_ylim([0, 1])

    # 4. AUROC Improvement
    ax4 = fig.add_subplot(gs[1, 1])
    colors_roc = ['darkgreen' if x > 0 else 'darkred' for x in df_metrics['delta_AUROC']]
    bars_roc = ax4.bar(df_metrics['fold'], df_metrics['delta_AUROC'],
                       color=colors_roc, alpha=0.7, edgecolor='black', linewidth=1.5)
    ax4.axhline(0, color='black', linestyle='--', linewidth=1)
    ax4.set_ylabel('Δ AUROC (LSTM - Baseline)', fontsize=11, fontweight='bold')
    ax4.set_xlabel('Fold', fontsize=11, fontweight='bold')
    ax4.set_title('AUROC Improvement by Fold', fontsize=13, fontweight='bold')
    ax4.grid(axis='y', alpha=0.3)
    
    # Add value labels
    for bar, val in zip(bars_roc, df_metrics['delta_AUROC']):
        height = bar.get_height()
        ax4.text(bar.get_x() + bar.get_width()/2., height,
                f'{val:+.4f}', ha='center', va='bottom' if val > 0 else 'top',
                fontsize=9, fontweight='bold')

    plt.savefig(output_dir / 'lstm_comprehensive_analysis.png', dpi=300, bbox_inches='tight')
    print(f"\n Saved: {output_dir / 'lstm_comprehensive_analysis.png'}")
    plt.close()
    
    # Save metrics to CSV
    df_metrics.to_csv(output_dir / 'lstm_fold_metrics.csv', index=False)
    print(f" Saved: {output_dir / 'lstm_fold_metrics.csv'}")




# LSTM FUNCTIONS END




def conf_weighted_then_multiscale(df_per_seed, sigma=1.0, conf_thr=0.1, sigmas_multiscale=[0.5, 1.0, 1.5, 2.0]):
    """
    Combination 4: Confidence-weighted smoothing followed by Multiscale Temporal Ensemble.
    - Stage 1: local adaptive smoothing based on prediction confidence
    - Stage 2: global temporal smoothing at multiple scales
    """
    # Stage 1: Confidence-weighted smoothing
    df_per_seed = align_seed_frames(df_per_seed)
    probs_all = np.array([df_per_seed[s]["p_defective"].values for s in df_per_seed])
    avg_probs, std_probs = np.mean(probs_all, axis=0), np.std(probs_all, axis=0)
    df_ens = list(df_per_seed.values())[0].copy()
    df_ens["p_defective"], df_ens["p_std"] = avg_probs, std_probs

    conf_smoothed = np.zeros(len(df_ens))
    for probs, _, idxs in build_sequences(df_ens):
        if len(probs) < 3:
            conf_smoothed[idxs] = probs
            continue
        seq_stds = np.array([df_ens.loc[i, "p_std"] for i in idxs])
        smooth_seq = gaussian_filter1d(probs, sigma=sigma, mode="nearest")
        conf_w = 1.0 - np.clip(seq_stds / conf_thr, 0, 1)
        conf_smoothed[idxs] = conf_w * probs + (1 - conf_w) * smooth_seq

    # Stage 2: Multiscale smoothing on top of confidence-weighted output
    df_stage2 = df_ens.assign(p_defective=conf_smoothed)
    multi_probs = []
    for s in sigmas_multiscale:
        s_probs = np.zeros(len(df_stage2))
        for probs, _, idxs in build_sequences(df_stage2):
            sm = gaussian_filter1d(probs, sigma=s, mode="nearest") if len(probs) >= 3 else probs
            s_probs[idxs] = sm
        multi_probs.append(s_probs)

    final_probs = np.mean(multi_probs, axis=0)
    preds = (final_probs >= 0.5).astype(int)
    return compute_metrics(df_ens["y_true_defective"], preds, final_probs)


def tune_conf_weighted_then_multiscale(train_df_per_seed):
    print("\n Grid search for Confidence-Weighted + Multiscale combo:")
    candidates = []
    for sigma in SIGMA_GRID:
        for thr in CONF_THRESH_GRID:
            for sigmas_ms in MULTISCALE_SIGMA_SETS:
                m = conf_weighted_then_multiscale(train_df_per_seed, sigma=sigma, conf_thr=thr, sigmas_multiscale=sigmas_ms)
                print(f"    σ_conf={sigma:<3}, thr={thr:<4}, sigmas_ms={sigmas_ms} → "
                      f"AUPRC={m['auprc']:.4f}, F1={m['f1']:.4f}, "
                      f"Prec={m['precision']:.4f}, Rec={m['recall']:.4f}")
                candidates.append(((sigma, thr, sigmas_ms), m))
    best_params, best_metrics = select_best_model(candidates)
    sigma_best, thr_best, sigmas_best = best_params
    print(f"  Best train (conf-weighted+multiscale): σ={sigma_best}, thr={thr_best}, sigmas={sigmas_best}, "
          f"Prec={best_metrics['precision']:.4f}, Rec={best_metrics['recall']:.4f}, "
          f"F1={best_metrics['f1']:.4f}")
    return best_params

def conf_weighted_then_isolated_removal(df_per_seed, sigma, conf_thr, context=1):
    """
    Combination: Confidence-weighted smoothing followed by isolated removal.
    Smooth predictions based on uncertainty, then remove isolated defect frames.
    """
    # First: apply confidence-weighted smoothing
    df_per_seed = align_seed_frames(df_per_seed)
    probs_all = np.array([df_per_seed[s]["p_defective"].values for s in df_per_seed])
    avg_probs, std_probs = np.mean(probs_all, axis=0), np.std(probs_all, axis=0)
    df_ens = list(df_per_seed.values())[0].copy()
    df_ens["p_defective"], df_ens["p_std"] = avg_probs, std_probs

    smoothed = np.zeros(len(df_ens))
    for probs, _, idxs in build_sequences(df_ens):
        if len(probs) < 3:
            smoothed[idxs] = probs
            continue
        seq_stds = np.array([df_ens.loc[i, "p_std"] for i in idxs])
        smooth_seq = gaussian_filter1d(probs, sigma=sigma, mode="nearest")
        conf_w = 1.0 - np.clip(seq_stds / conf_thr, 0, 1)
        smoothed[idxs] = conf_w * probs + (1 - conf_w) * smooth_seq

    # Second: apply isolated removal on the smoothed output
    cleaned_probs = np.zeros(len(df_ens))
    for probs, _, idxs in build_sequences(df_ens.assign(p_defective=smoothed)):
        preds = (probs >= 0.5).astype(int)
        filtered = preds.copy()
        for i in range(len(preds)):
            if preds[i] == 1:
                start = max(0, i - context)
                end = min(len(preds), i + context + 1)
                neighbors = np.delete(preds[start:end], i - start)
                if np.sum(neighbors) == 0:
                    filtered[i] = 0
        cleaned_probs[idxs] = filtered

    preds = (cleaned_probs >= 0.5).astype(int)
    return compute_metrics(df_ens["y_true_defective"], preds, cleaned_probs)

def tune_conf_weighted_then_isolated(train_df_per_seed):
    print("\n Grid search for Confidence-Weighted + Isolated Removal combo:")
    candidates = []
    for sigma in SIGMA_GRID:
        for thr in CONF_THRESH_GRID:
            for ctx in [1, 2, 3, 4, 5]:
                m = conf_weighted_then_isolated_removal(train_df_per_seed, sigma, thr, context=ctx)
                print(f"    σ={sigma:<3}, thr={thr:<4}, ctx={ctx} → "
                      f"AUPRC={m['auprc']:.4f}, F1={m['f1']:.4f}, "
                      f"Prec={m['precision']:.4f}, Rec={m['recall']:.4f}")
                candidates.append(((sigma, thr, ctx), m))
    best_params, best_metrics = select_best_model(candidates)
    sigma, thr, ctx = best_params
    print(f" Best train (conf-w+iso): σ={sigma}, thr={thr}, ctx={ctx}, "
          f"Prec={best_metrics['precision']:.4f}, Rec={best_metrics['recall']:.4f}, "
          f"F1={best_metrics['f1']:.4f}")
    return best_params

def multiscale_then_defect_voting(df_per_seed, sigmas, window=5, conf_thr=0.1, alpha=0.6):
    """
    Combination: Multiscale temporal smoothing followed by Defect Voting.
    - Multiscale smooths predictions at multiple temporal resolutions.
    - Defect Voting enforces local majority consistency.
    - Confidence weighting (conf_thr) adaptively controls how much to trust the local voting.
    - alpha controls the overall balance between model vs. local voting.
    """
    # Step 1: Multiscale smoothing
    df_per_seed = align_seed_frames(df_per_seed)
    probs_all = np.array([df_per_seed[s]["p_defective"].values for s in df_per_seed])
    avg_probs = np.mean(probs_all, axis=0)
    df_ens = list(df_per_seed.values())[0].copy()
    df_ens["p_defective"] = avg_probs

    multi_probs = []
    for sigma in sigmas:
        s_probs = np.zeros(len(df_ens))
        for probs, _, idxs in build_sequences(df_ens):
            sm = gaussian_filter1d(probs, sigma=sigma, mode="nearest") if len(probs) >= 3 else probs
            s_probs[idxs] = sm
        multi_probs.append(s_probs)
    smoothed = np.mean(multi_probs, axis=0)

    # Step 2: Defect voting on smoothed output
    voted_probs = np.zeros(len(df_ens))
    half_w = window // 2
    df_sm = df_ens.assign(p_defective=smoothed)

    for probs, _, idxs in build_sequences(df_sm):
        seq_votes = np.zeros_like(probs)
        seq_stds = np.zeros_like(probs)

        for t in range(len(probs)):
            start = max(0, t - half_w)
            end = min(len(probs), t + half_w + 1)
            local = probs[start:end]
            local_bin = (local >= 0.5).astype(int)

            # Majority vote in local window
            seq_votes[t] = np.mean(local_bin)
            # Local uncertainty (temporal std of probabilities)
            seq_stds[t] = local.std()

        # Compute adaptive confidence weight
        conf_w = 1.0 - np.clip(seq_stds / conf_thr, 0, 1)
        # Blend model vs. voting with global + local weighting
        voted_probs[idxs] = (alpha * conf_w) * probs + (1 - alpha * conf_w) * seq_votes

    preds = (voted_probs >= 0.5).astype(int)
    return compute_metrics(df_ens["y_true_defective"], preds, voted_probs)




def tune_multiscale_then_voting(train_df_per_seed):
    print("\n Grid search for Multiscale + Defect Voting combo:")
    candidates = []
    for sigmas in MULTISCALE_SIGMA_SETS:
        for window in [3, 5, 7, 9]:
            for thr in CONF_THRESH_GRID:
                m = multiscale_then_defect_voting(train_df_per_seed, sigmas, window=window, conf_thr=thr)
                print(f"    sigmas={sigmas}, window={window:<2}, thr={thr:<4} → "
                      f"AUPRC={m['auprc']:.4f}, F1={m['f1']:.4f}, "
                      f"Prec={m['precision']:.4f}, Rec={m['recall']:.4f}")
                candidates.append(((sigmas, window, thr), m))

    best_params, best_metrics = select_best_model(candidates)
    sigmas_best, window_best, thr_best = best_params
    print(f"  Best train (multi+vote): sigmas={sigmas_best}, window={window_best}, thr={thr_best}, "
          f"Prec={best_metrics['precision']:.4f}, Rec={best_metrics['recall']:.4f}, "
          f"F1={best_metrics['f1']:.4f}")
    return best_params

def conf_median_then_min_cluster(df_per_seed, kernel_size=5, conf_thr=0.1, min_cluster_size=3):
    """
    Combination: Confidence-Weighted Median smoothing followed by Min Cluster Size filtering.
    Smooths noisy probabilities and removes short defect clusters.
    """
    #Step 1: Confidence-weighted median smoothing
    df_per_seed = align_seed_frames(df_per_seed)
    probs_all = np.array([df_per_seed[s]["p_defective"].values for s in df_per_seed])
    avg_probs, std_probs = np.mean(probs_all, axis=0), np.std(probs_all, axis=0)
    df_ens = list(df_per_seed.values())[0].copy()
    df_ens["p_defective"], df_ens["p_std"] = avg_probs, std_probs

    smoothed = np.zeros(len(df_ens))
    for probs, _, idxs in build_sequences(df_ens):
        if len(probs) < 3:
            smoothed[idxs] = probs
            continue
        seq_stds = np.array([df_ens.loc[i, "p_std"] for i in idxs])
        sm = medfilt(probs, kernel_size=kernel_size)
        conf_w = 1.0 - np.clip(seq_stds / conf_thr, 0, 1)
        smoothed[idxs] = conf_w * probs + (1 - conf_w) * sm

    # Step 2: Min cluster size filtering on smoothed output 
    filtered_probs = np.zeros(len(df_ens))
    for probs, _, idxs in build_sequences(df_ens.assign(p_defective=smoothed)):
        seq_len = len(probs)
        min_cluster_size = min_cluster_size

        preds = (probs >= 0.5).astype(int)
        filtered = preds.copy()
        start = 0
        for val, group in groupby(preds):
            length = len(list(group))
            if val == 1 and length < min_cluster_size:
                filtered[start:start + length] = 0
            start += length
        filtered_probs[idxs] = filtered

    preds = (filtered_probs >= 0.5).astype(int)
    return compute_metrics(df_ens["y_true_defective"], preds, filtered_probs)
def tune_conf_median_then_min_cluster(train_df_per_seed):
    print("\n  🔍 Grid search for Confidence-Weighted Median + Min Cluster combo:")
    kernel_grid = [3, 5, 7, 9]
    conf_grid = CONF_THRESH_GRID
    cluster_grid = [2, 3, 4, 5, 6, 7]
    candidates = []
    for k in kernel_grid:
        for thr in conf_grid:
            for size in cluster_grid:
                m = conf_median_then_min_cluster(train_df_per_seed, kernel_size=k, conf_thr=thr, min_cluster_size=size)
                print(f"    kernel={k:<2}, thr={thr:<4}, min_cluster={size:<2} → "
                      f"AUPRC={m['auprc']:.4f}, F1={m['f1']:.4f}, "
                      f"Prec={m['precision']:.4f}, Rec={m['recall']:.4f}")
                candidates.append(((k, thr, size), m))
    best_params, best_metrics = select_best_model(candidates)
    k_best, thr_best, size_best = best_params
    print(f"  Best train (conf-median+mincluster): kernel={k_best}, thr={thr_best}, min_cluster={size_best}, "
          f"Prec={best_metrics['precision']:.4f}, Rec={best_metrics['recall']:.4f}, "
          f"F1={best_metrics['f1']:.4f}")
    return best_params

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
# Selection helper 
def select_best_model(candidates):
    """
    candidates: list of (params, metrics)
    Returns the params that maximize precision subject to recall ≥ 0.75,
    otherwise the params with best F1.
    """
    # Filter by recall boundary
    valid = [(p, m) for p, m in candidates if m["recall"] >= 0.75]
    if valid:
        # pick highest precision among those
        best = max(valid, key=lambda x: x[1]["precision"])
    else:
        # fallback: best F1
        best = max(candidates, key=lambda x: x[1]["f1"])
    return best


def extract_frame_number(image_id):
    try:
        return int(str(image_id).split('_')[0])
    except Exception:
        return 0

def compute_metrics(y_true, y_pred, y_prob, return_predictions=False):
    m = {}
    m["accuracy"] = accuracy_score(y_true, y_pred)
    m["balanced_accuracy"] = balanced_accuracy_score(y_true, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, pos_label=1, average="binary", zero_division=0
    )
    m.update({"precision": prec, "recall": rec, "f1": f1})
    
    # Only compute AUPRC if y_prob contains actual probabilities
    if len(np.unique(y_prob)) > 2:  # Not binary
        try:
            m["auroc"] = roc_auc_score(y_true, y_prob)
            m["auprc"] = average_precision_score(y_true, y_prob)
        except Exception:
            m["auroc"], m["auprc"] = np.nan, np.nan
    else:
        m["auroc"], m["auprc"] = np.nan, np.nan
    
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel() if cm.size == 4 else (0, 0, 0, 0)
    m.update({"tp": int(tp), "tn": int(tn), "fp": int(fp), "fn": int(fn)})
    
    # NEW: Add normalized confusion matrix
    cm_norm = confusion_matrix(y_true, y_pred, labels=[0, 1], normalize='true')
    m['cm_normalized'] = cm_norm.tolist()
    
    # NEW: Return predictions if requested
    if return_predictions:
        return m, {'probs': y_prob, 'labels': y_pred, 'y_true': y_true}
    return m



def save_fold_predictions(df_val_per_seed, method_results, fold_name, output_dir):
    """Save all method predictions for this fold."""
    pred_dir = output_dir / "predictions"
    pred_dir.mkdir(exist_ok=True)
    
    df_base = list(df_val_per_seed.values())[0].copy()
    
    # Save each method's predictions
    for method_name, pred_data in method_results.items():
        if pred_data is not None:
            df_pred = df_base.copy()
            df_pred['pred_probs'] = pred_data['probs']
            df_pred['pred_labels'] = pred_data['labels']
            
            pred_file = pred_dir / f"{fold_name}_{method_name}.csv"
            df_pred.to_csv(pred_file, index=False)
    
    print(f" Saved predictions for {len(method_results)} methods")
    
def print_metrics(name, m):
    print(f"\n{name}")
    print(f"  AUPRC={m['auprc']:.4f} | F1={m['f1']:.4f} | "
          f"Prec={m['precision']:.4f} | Rec={m['recall']:.4f}")
    print(f"  TP={m['tp']}, TN={m['tn']}, FP={m['fp']}, FN={m['fn']}")

def load_predictions(pred_csv, img_dir):
    df = pd.read_csv(pred_csv)
    parsed = []
    for f in df["filename"]:
        m = FILENAME_PATTERN.match(Path(f).name)
        if not m:
            raise ValueError(f"Filename pattern failed: {filename}")
        parsed.append(m.groupdict() if m else {})
    df = pd.concat([df, pd.DataFrame(parsed)], axis=1)
    df["path"] = df["filename"].apply(lambda f: str(Path(img_dir) / f))
    return df

def build_sequences(df):
    seqs = []
    for _, grp in df.groupby(["station", "track", "side"], dropna=False):
        frames = grp["image_id"].map(extract_frame_number)
        grp_sorted = grp.iloc[frames.argsort()]
        seqs.append((
            grp_sorted["p_defective"].values,
            grp_sorted["y_true_defective"].astype(int).values,
            grp_sorted.index.values
        ))
    return seqs

# ============================================================================
# METHODS
# ============================================================================
def align_seed_frames(df_per_seed):
    # Sort or merge so that all seeds have identical row order
    return {s: df.sort_values("filename").reset_index(drop=True) for s, df in df_per_seed.items()}

def baseline_ensemble(df_per_seed):
    df_per_seed = align_seed_frames(df_per_seed)  
    probs = np.mean([df_per_seed[s]["p_defective"].values for s in df_per_seed], axis=0)
    preds = (probs >= 0.5).astype(int)
    df_base = list(df_per_seed.values())[0]
    return compute_metrics(df_base["y_true_defective"], preds, probs)



def confidence_weighted_smoothing(df_per_seed, sigma, confidence_threshold):
    df_per_seed = align_seed_frames(df_per_seed)
    probs_all = np.array([df_per_seed[s]["p_defective"].values for s in df_per_seed])
    avg_probs, std_probs = np.mean(probs_all, axis=0), np.std(probs_all, axis=0)
    df_ens = list(df_per_seed.values())[0].copy()
    df_ens["p_defective"], df_ens["p_std"] = avg_probs, std_probs

    smoothed = np.zeros(len(df_ens))
    for probs, _, idxs in build_sequences(df_ens):
        if len(probs) < 3:
            smoothed[idxs] = probs
            continue
        seq_stds = np.array([df_ens.loc[i, "p_std"] for i in idxs])
        smooth_seq = gaussian_filter1d(probs, sigma=sigma, mode="nearest")
        conf_w = 1.0 - np.clip(seq_stds / confidence_threshold, 0, 1)
        smoothed[idxs] = conf_w * probs + (1 - conf_w) * smooth_seq

    preds = (smoothed >= 0.5).astype(int)
    return compute_metrics(df_ens["y_true_defective"], preds, smoothed)

def confidence_weighted_median(df_per_seed, kernel_size=5, conf_thr=0.1):
    """Apply median filtering with confidence weighting per sequence."""
    df_per_seed = align_seed_frames(df_per_seed)
    probs_all = np.array([df_per_seed[s]["p_defective"].values for s in df_per_seed])
    avg_probs, std_probs = np.mean(probs_all, axis=0), np.std(probs_all, axis=0)
    df_ens = list(df_per_seed.values())[0].copy()
    df_ens["p_defective"], df_ens["p_std"] = avg_probs, std_probs

    smoothed = np.zeros(len(df_ens))
    for probs, _, idxs in build_sequences(df_ens):
        if len(probs) < 3:
            smoothed[idxs] = probs
            continue
        seq_stds = np.array([df_ens.loc[i, "p_std"] for i in idxs])
        sm = medfilt(probs, kernel_size=kernel_size)
        conf_w = 1.0 - np.clip(seq_stds / conf_thr, 0, 1)
        smoothed[idxs] = conf_w * probs + (1 - conf_w) * sm

    preds = (smoothed >= 0.5).astype(int)
    return compute_metrics(df_ens["y_true_defective"], preds, smoothed)



def multiscale_temporal_ensemble(df_per_seed, sigmas):
    df_per_seed = align_seed_frames(df_per_seed)
    probs_all = np.array([df_per_seed[s]["p_defective"].values for s in df_per_seed])
    avg_probs = np.mean(probs_all, axis=0)
    df_ens = list(df_per_seed.values())[0].copy()
    df_ens["p_defective"] = avg_probs

    multi_probs = []
    for sigma in sigmas:
        s_probs = np.zeros(len(df_ens))
        for probs, _, idxs in build_sequences(df_ens):
            sm = gaussian_filter1d(probs, sigma=sigma, mode="nearest") if len(probs) >= 3 else probs
            s_probs[idxs] = sm
        multi_probs.append(s_probs)
    final = np.mean(multi_probs, axis=0)
    preds = (final >= 0.5).astype(int)
    return compute_metrics(df_ens["y_true_defective"], preds, final)

def defect_voting(df_per_seed, window=5, conf_thr=0.1):
    """
    Defect Voting ensemble:
    - Smooths the binary decisions within each sequence by majority voting.
    - Uses confidence weighting (lower std → higher weight).
    """
    df_per_seed = align_seed_frames(df_per_seed)
    probs_all = np.array([df_per_seed[s]["p_defective"].values for s in df_per_seed])
    avg_probs, std_probs = np.mean(probs_all, axis=0), np.std(probs_all, axis=0)
    df_ens = list(df_per_seed.values())[0].copy()
    df_ens["p_defective"], df_ens["p_std"] = avg_probs, std_probs

    voted_probs = np.zeros(len(df_ens))
    half_w = window // 2

    for probs, _, idxs in build_sequences(df_ens):
        seq_stds = np.array([df_ens.loc[i, "p_std"] for i in idxs])
        seq_votes = np.zeros_like(probs)
        for t in range(len(probs)):
            start = max(0, t - half_w)
            end = min(len(probs), t + half_w + 1)
            local = probs[start:end]
            local_bin = (local >= 0.5).astype(int)
            seq_votes[t] = np.mean(local_bin)  # proportion of "defect" votes
        conf_w = 1.0 - np.clip(seq_stds / conf_thr, 0, 1)
        voted_probs[idxs] = conf_w * probs + (1 - conf_w) * seq_votes

    preds = (voted_probs >= 0.5).astype(int)
    return compute_metrics(df_ens["y_true_defective"], preds, voted_probs)


def min_cluster_size_filter(df_per_seed, min_cluster_size=3):
    """
    Enforce minimum cluster size on predicted defects.
    Clusters of consecutive defect frames shorter than min_cluster_size are removed.
    """
    df_per_seed = align_seed_frames(df_per_seed)
    probs_all = np.array([df_per_seed[s]["p_defective"].values for s in df_per_seed])
    avg_probs = np.mean(probs_all, axis=0)
    df_ens = list(df_per_seed.values())[0].copy()
    df_ens["p_defective"] = avg_probs

    filtered_probs = np.zeros(len(df_ens))
    for probs, _, idxs in build_sequences(df_ens):
        seq_len = len(probs)
        min_cluster_size = min_cluster_size

        preds = (probs >= 0.5).astype(int)
        filtered = preds.copy()
        # Identify clusters of consecutive identical values
        start = 0
        for val, group in groupby(preds):
            length = len(list(group))
            if val == 1 and length < min_cluster_size:
                filtered[start:start+length] = 0
            start += length
        filtered_probs[idxs] = filtered
    preds = (filtered_probs >= 0.5).astype(int)
    return compute_metrics(df_ens["y_true_defective"], preds, filtered_probs)

def isolated_removal(df_per_seed, context=1):
    """
    Remove isolated defect frames (those not supported by nearby positives).
    context: how many neighboring frames to check on each side.
    """
    df_per_seed = align_seed_frames(df_per_seed)
    probs_all = np.array([df_per_seed[s]["p_defective"].values for s in df_per_seed])
    avg_probs = np.mean(probs_all, axis=0)
    df_ens = list(df_per_seed.values())[0].copy()
    df_ens["p_defective"] = avg_probs

    cleaned_probs = np.zeros(len(df_ens))
    for probs, _, idxs in build_sequences(df_ens):
        preds = (probs >= 0.5).astype(int)
        filtered = preds.copy()
        for i in range(len(preds)):
            if preds[i] == 1:
                # Check local context
                start = max(0, i - context)
                end = min(len(preds), i + context + 1)
                neighbors = np.delete(preds[start:end], i - start)
                if np.sum(neighbors) == 0:
                    filtered[i] = 0
        cleaned_probs[idxs] = filtered
    preds = (cleaned_probs >= 0.5).astype(int)
    return compute_metrics(df_ens["y_true_defective"], preds, cleaned_probs)

def temporal_smoothing(df_per_seed, window=5):
    """
    Apply uniform moving average smoothing across each sequence.
    """
    df_per_seed = align_seed_frames(df_per_seed)
    probs_all = np.array([df_per_seed[s]["p_defective"].values for s in df_per_seed])
    avg_probs = np.mean(probs_all, axis=0)
    df_ens = list(df_per_seed.values())[0].copy()
    df_ens["p_defective"] = avg_probs

    smoothed_probs = np.zeros(len(df_ens))
    half_w = window // 2

    for probs, _, idxs in build_sequences(df_ens):
        smooth_seq = np.zeros_like(probs)
        for i in range(len(probs)):
            start = max(0, i - half_w)
            end = min(len(probs), i + half_w + 1)
            smooth_seq[i] = np.mean(probs[start:end])
        smoothed_probs[idxs] = smooth_seq

    preds = (smoothed_probs >= 0.5).astype(int)
    return compute_metrics(df_ens["y_true_defective"], preds, smoothed_probs)


def tune_confidence_weighted(train_df_per_seed):
    print("\n Grid search for Confidence-Weighted smoothing:")
    candidates = []
    for sigma in SIGMA_GRID:
        for thr in CONF_THRESH_GRID:
            m = confidence_weighted_smoothing(train_df_per_seed, sigma, thr)
            print(f"    σ={sigma:<3}, thr={thr:<4} → "
                  f"AUPRC={m['auprc']:.4f}, F1={m['f1']:.4f}, "
                  f"Prec={m['precision']:.4f}, Rec={m['recall']:.4f}")
            candidates.append(((sigma, thr), m))

    best_params, best_metrics = select_best_model(candidates)
    sigma, thr = best_params
    print(f" Best train (conf-w): σ={sigma}, thr={thr}, "
          f"Prec={best_metrics['precision']:.4f}, Rec={best_metrics['recall']:.4f}, "
          f"F1={best_metrics['f1']:.4f}")
    return best_params

def tune_confidence_weighted_median(train_df_per_seed):
    print("\n Grid search for Confidence-Weighted Median smoothing:")
    kernel_sizes = [3, 5, 7, 9]
    conf_thresholds = [0.05, 0.1, 0.2]
    candidates = []
    for k in kernel_sizes:
        for thr in conf_thresholds:
            m = confidence_weighted_median(train_df_per_seed, kernel_size=k, conf_thr=thr)
            print(f"    kernel={k:<2}, thr={thr:<4} → "
                  f"AUPRC={m['auprc']:.4f}, F1={m['f1']:.4f}, "
                  f"Prec={m['precision']:.4f}, Rec={m['recall']:.4f}")
            candidates.append(((k, thr), m))

    best_params, best_metrics = select_best_model(candidates)
    k, thr = best_params
    print(f" Best train (conf-median): kernel={k}, thr={thr}, "
          f"Prec={best_metrics['precision']:.4f}, Rec={best_metrics['recall']:.4f}, "
          f"F1={best_metrics['f1']:.4f}")
    return best_params

def tune_multiscale(train_df_per_seed):
    print("\n Grid search for Multiscale Temporal smoothing:")
    candidates = []
    for sigmas in MULTISCALE_SIGMA_SETS:
        m = multiscale_temporal_ensemble(train_df_per_seed, sigmas)
        print(f"    sigmas={sigmas} → "
              f"AUPRC={m['auprc']:.4f}, F1={m['f1']:.4f}, "
              f"Prec={m['precision']:.4f}, Rec={m['recall']:.4f}")
        candidates.append((sigmas, m))

    best_params, best_metrics = select_best_model(candidates)
    print(f" Best train (multiscale): {best_params}, "
          f"Prec={best_metrics['precision']:.4f}, Rec={best_metrics['recall']:.4f}, "
          f"F1={best_metrics['f1']:.4f}")
    return best_params
def tune_defect_voting(train_df_per_seed):
    print("\n Grid search for Defect Voting:")
    window_grid = [3, 5, 7, 9]
    conf_thr_grid = [0.05, 0.1, 0.2]
    candidates = []

    for w in window_grid:
        for thr in conf_thr_grid:
            m = defect_voting(train_df_per_seed, window=w, conf_thr=thr)
            print(f"    window={w:<2}, thr={thr:<4} → "
                  f"AUPRC={m['auprc']:.4f}, F1={m['f1']:.4f}, "
                  f"Prec={m['precision']:.4f}, Rec={m['recall']:.4f}")
            candidates.append(((w, thr), m))

    best_params, best_metrics = select_best_model(candidates)
    w_best, thr_best = best_params
    print(f" Best train (defect voting): window={w_best}, thr={thr_best}, "
          f"Prec={best_metrics['precision']:.4f}, Rec={best_metrics['recall']:.4f}, "
          f"F1={best_metrics['f1']:.4f}")
    return best_params
def tune_min_cluster_size(train_df_per_seed):
    print("\n Grid search for Min Cluster Size filtering:")
    cluster_sizes = [2, 3, 4, 5, 6, 7]
    candidates = []
    for size in cluster_sizes:
        m = min_cluster_size_filter(train_df_per_seed, min_cluster_size=size)
        print(f"    min_cluster_size={size:<2} → "
              f"AUPRC={m['auprc']:.4f}, F1={m['f1']:.4f}, "
              f"Prec={m['precision']:.4f}, Rec={m['recall']:.4f}")
        candidates.append(((size,), m))

    best_params, best_metrics = select_best_model(candidates)
    size = best_params[0]
    print(f" Best train (min-cluster): size={size}, "
          f"Prec={best_metrics['precision']:.4f}, Rec={best_metrics['recall']:.4f}, "
          f"F1={best_metrics['f1']:.4f}")
    return size


def tune_isolated_removal(train_df_per_seed):
    print("\n Grid search for Isolated Removal:")
    context_sizes = [1, 2, 3, 4, 5]
    candidates = []
    for ctx in context_sizes:
        m = isolated_removal(train_df_per_seed, context=ctx)
        print(f"    context={ctx:<2} → "
              f"AUPRC={m['auprc']:.4f}, F1={m['f1']:.4f}, "
              f"Prec={m['precision']:.4f}, Rec={m['recall']:.4f}")
        candidates.append(((ctx,), m))

    best_params, best_metrics = select_best_model(candidates)
    ctx_best = best_params[0]
    print(f"  Best train (isolated removal): context={ctx_best}, "
          f"Prec={best_metrics['precision']:.4f}, Rec={best_metrics['recall']:.4f}, "
          f"F1={best_metrics['f1']:.4f}")
    return ctx_best

def tune_temporal_smoothing(train_df_per_seed):
    print("\n Grid search for Temporal Smoothing:")
    window_grid = [1,2,3]
    candidates = []
    for w in window_grid:
        m = temporal_smoothing(train_df_per_seed, window=w)
        print(f"    window={w:<2} → "
              f"AUPRC={m['auprc']:.4f}, F1={m['f1']:.4f}, "
              f"Prec={m['precision']:.4f}, Rec={m['recall']:.4f}")
        candidates.append(((w,), m))

    best_params, best_metrics = select_best_model(candidates)
    w_best = best_params[0]
    print(f" Best train (temporal smoothing): window={w_best}, "
          f"Prec={best_metrics['precision']:.4f}, Rec={best_metrics['recall']:.4f}, "
          f"F1={best_metrics['f1']:.4f}")
    return w_best
def baseline_single(df_val_per_seed):
    """
    Compute baseline metrics for each seed individually (non-ensemble) and average them.
    """
    all_metrics = []
    for seed, df in df_val_per_seed.items():
        probs = df["p_defective"].values
        preds = (probs >= 0.5).astype(int)
        m = compute_metrics(df["y_true_defective"], preds, probs)
        m["seed"] = seed
        all_metrics.append(m)
    df_res = pd.DataFrame(all_metrics)
    return df_res.mean(numeric_only=True).to_dict(), df_res.std(numeric_only=True).to_dict()


def select_best_method_by_precision(results_df, baseline_method="baseline_ensemble", 
                                    recall_threshold_mode='maintain'):
    """
    Select best method prioritizing precision while maintaining recall.
    
    Args:
        results_df: DataFrame with all results
        baseline_method: Name of baseline method
        recall_threshold_mode: 
            - 'maintain': recall >= baseline_recall * 0.95
            - 'absolute': recall >= 0.75 ( original criterion)
            - 'flexible': recall >= baseline_recall * 0.90
    
    Returns:
        best_method_name, category, full_ranking_df
    """
   
    print("SELECTING BEST METHOD (PRECISION-FIRST STRATEGY)")
    
    
    # Get baseline performance
    baseline = results_df[results_df["method"] == baseline_method]
    baseline_metrics = baseline[["auprc", "precision", "recall", "f1"]].mean()
    
    print(f"\n Baseline ({baseline_method}):")
    print(f"   Precision: {baseline_metrics['precision']:.4f}")
    print(f"   Recall: {baseline_metrics['recall']:.4f}")
    print(f"   F1: {baseline_metrics['f1']:.4f}")
    print(f"   AUPRC: {baseline_metrics['auprc']:.4f}")
    
    # Compute mean metrics per method
    non_baseline = results_df[results_df['method'] != baseline_method]
    method_summary = non_baseline.groupby('method').agg({
        'auprc': ['mean', 'std'],
        'precision': ['mean', 'std'],
        'recall': ['mean', 'std'],
        'f1': ['mean', 'std'],
        'tp': 'mean',
        'fp': 'mean',
        'tn': 'mean',
        'fn': 'mean'
    }).reset_index()
    
    # Flatten columns
    method_summary.columns = ['method'] + [
        f"{col[0]}_{col[1]}" if col[1] else col[0] 
        for col in method_summary.columns[1:]
    ]
    
    # Set recall threshold based on mode
    if recall_threshold_mode == 'maintain':
        recall_threshold = baseline_metrics['recall'] * 0.95  # Allow max 5% drop
        threshold_label = f"{recall_threshold:.4f} (95% of baseline)"
    elif recall_threshold_mode == 'absolute':
        recall_threshold = 0.75
        threshold_label = "0.75 (absolute)"
    else:  # flexible
        recall_threshold = baseline_metrics['recall'] * 0.90  # Allow max 10% drop
        threshold_label = f"{recall_threshold:.4f} (90% of baseline)"
    
    print(f"\n Selection Criteria:")
    print(f"   1. Recall >= {threshold_label}")
    print(f"   2. Among those: maximize Precision")
    print(f"   3. Tiebreaker: F1 score")
    
    # Filter candidates by recall threshold
    candidates = method_summary[method_summary['recall_mean'] >= recall_threshold].copy()
    
    print(f"\n Methods meeting recall threshold: {len(candidates)} / {len(method_summary)}")
    
    if len(candidates) == 0:
        print(f"\n No methods meet recall threshold!")
        print(f"   Falling back to best F1 score...")
        candidates = method_summary.copy()
        best = candidates.sort_values('f1_mean', ascending=False).iloc[0]
        warning_flag = True
    else:
        # Sort by precision (descending), then F1 (descending)
        candidates = candidates.sort_values(['precision_mean', 'f1_mean'], ascending=False)
        best = candidates.iloc[0]
        warning_flag = False
    
    # Calculate improvements
    prec_gain = best['precision_mean'] - baseline_metrics['precision']
    rec_gain = best['recall_mean'] - baseline_metrics['recall']
    f1_gain = best['f1_mean'] - baseline_metrics['f1']
    auprc_gain = best['auprc_mean'] - baseline_metrics['auprc'] if not np.isnan(best['auprc_mean']) else np.nan
    
    # Determine category
    if np.isnan(best['auprc_mean']):
        category = 'binary_only'
        category_label = "Binary-only (no AUPRC)"
    else:
        category = 'probability_based'
        category_label = " Probability-based"
    
    print(f"\n{'WARN' if warning_flag else ' OK '}BEST METHOD: {best['method']}")
    print(f"   Category: {category_label}")
    print(f"   Precision: {best['precision_mean']:.4f} ± {best['precision_std']:.4f} (Δ = {prec_gain:+.4f}, {prec_gain/baseline_metrics['precision']*100:+.1f}%)")
    print(f"   Recall:    {best['recall_mean']:.4f} ± {best['recall_std']:.4f} (Δ = {rec_gain:+.4f}, {rec_gain/baseline_metrics['recall']*100:+.1f}%)")
    print(f"   F1:        {best['f1_mean']:.4f} ± {best['f1_std']:.4f} (Δ = {f1_gain:+.4f}, {f1_gain/baseline_metrics['f1']*100:+.1f}%)")
    if not np.isnan(auprc_gain):
        print(f"   AUPRC:     {best['auprc_mean']:.4f} ± {best['auprc_std']:.4f} (Δ = {auprc_gain:+.4f}, {auprc_gain/baseline_metrics['auprc']*100:+.1f}%)")
    else:
        print(f"   AUPRC:     N/A (binary predictions)")
    
    print(f"\n   Confusion Matrix (avg):")
    print(f"   TP={best['tp_mean']:.1f}, FP={best['fp_mean']:.1f}")
    print(f"   FN={best['fn_mean']:.1f}, TN={best['tn_mean']:.1f}")
    
    # Show top 5 candidates
    print(f"\nTop 5 Candidates (by precision):")
    print(f"{'Rank':<5} {'Method':<45} {'Precision':<12} {'Recall':<12} {'F1':<12} {'AUPRC':<12}")
    print("-" * 100)
    for rank, (_, row) in enumerate(candidates.head(5).iterrows(), 1):
        auprc_str = f"{row['auprc_mean']:.4f}" if not np.isnan(row['auprc_mean']) else "N/A"
        print(f"{rank:<5} {row['method']:<45} {row['precision_mean']:.4f}      {row['recall_mean']:.4f}      {row['f1_mean']:.4f}      {auprc_str}")
    
    # Save detailed ranking
    candidates['precision_gain'] = candidates['precision_mean'] - baseline_metrics['precision']
    candidates['recall_gain'] = candidates['recall_mean'] - baseline_metrics['recall']
    candidates['f1_gain'] = candidates['f1_mean'] - baseline_metrics['f1']
    candidates.to_csv(OUTPUT_DIR / "method_ranking_by_precision.csv", index=False)
    print(f"\n Full ranking saved to: {OUTPUT_DIR / 'method_ranking_by_precision.csv'}")
    
    return best['method'], category, candidates, {
        'precision_gain': prec_gain,
        'recall_gain': rec_gain,
        'f1_gain': f1_gain,
        'auprc_gain': auprc_gain,
        'baseline_metrics': baseline_metrics.to_dict(),
        'best_metrics': {
            'precision': best['precision_mean'],
            'recall': best['recall_mean'],
            'f1': best['f1_mean'],
            'auprc': best['auprc_mean']
        }
    }


def print_method_ranking(results_df, category='all'):
    """
    Print ranked table of methods for thesis.
    
    Args:
        category: 'probability_based', 'binary_only', or 'all'
    """
    
    print(f"METHOD RANKING ({category.upper()})")

    
    if category == 'probability_based':
        df = results_df[results_df['auprc'].notna()].copy()
    elif category == 'binary_only':
        df = results_df[results_df['auprc'].isna()].copy()
    else:
        df = results_df.copy()
    
    # Compute mean metrics per method
    summary = df.groupby('method').agg({
        'auprc': ['mean', 'std'],
        'precision': ['mean', 'std'],
        'recall': ['mean', 'std'],
        'f1': ['mean', 'std']
    }).round(4)
    
    # Flatten column names
    summary.columns = ['_'.join(col).strip() for col in summary.columns.values]
    summary = summary.reset_index()
    
    # Sort by primary metric
    if 'auprc_mean' in summary.columns and summary['auprc_mean'].notna().any():
        summary = summary.sort_values('auprc_mean', ascending=False)
    else:
        summary = summary.sort_values('precision_mean', ascending=False)
    
    # Print table
    print("\n{:<45s} | {:>12s} | {:>12s} | {:>12s} | {:>12s}".format(
        "Method", "AUPRC", "Precision", "Recall", "F1"
    ))
    print("-" * 110)
    
    for idx, row in summary.iterrows():
        auprc_str = f"{row.get('auprc_mean', np.nan):.4f}±{row.get('auprc_std', np.nan):.4f}" if 'auprc_mean' in row else "N/A"
        prec_str = f"{row['precision_mean']:.4f}±{row['precision_std']:.4f}"
        rec_str = f"{row['recall_mean']:.4f}±{row['recall_std']:.4f}"
        f1_str = f"{row['f1_mean']:.4f}±{row['f1_std']:.4f}"
        
        print(f"{row['method']:<45s} | {auprc_str:>12s} | {prec_str:>12s} | {rec_str:>12s} | {f1_str:>12s}")
    
    return summary



## run main

In [ ]:
def main():
    print("EXP 4: PER-FOLD TUNING OF SEQUENCE-AWARE ENSEMBLES")
    set_seed(RANDOM_SEED)

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"Using device: {device}")

    fold_dirs = sorted([d for d in EXP3_ROOT.iterdir() if d.is_dir() and d.name.startswith("fold_")])
    all_results = []
    lstm_models = {} 
    val_data_per_fold = {}
    all_hyperparams = []  # NEW: Track all hyperparameters

    for fold in fold_dirs:
        
        fold_name = fold.name
        print(f"\n{'='*70}\nFOLD: {fold_name}\n{'='*70}")
        fold_dir = FOLDS_ROOT / fold_name
        train_dir, val_dir = fold_dir/"train", fold_dir/"val"

        seed_dirs = sorted([d for d in fold.iterdir() if d.is_dir() and d.name.startswith("seed")])
        seeds = [s.name for s in seed_dirs]
        if len(seed_dirs) == 0:
            print(f"No seeds found in {fold_name}, skipping.")
            continue
        df_train_per_seed, df_val_per_seed = {}, {}
        for seed in seeds:
            tr = fold/seed/"train_predictions.csv"
            va = fold/seed/"val_predictions.csv"
            if not (tr.exists() and va.exists()): continue
            df_tr = load_predictions(tr, train_dir)
            df_va = load_predictions(va, val_dir)
            for df in [df_tr, df_va]:
                df["p_defective"] = df["p_class_0"]
                df["y_true_defective"] = (df["y_true"] == 0).astype(int)
            df_train_per_seed[seed] = df_tr
            df_val_per_seed[seed] = df_va

        if len(df_val_per_seed) == 0 or len(df_train_per_seed) == 0:
            print(f"No valid seeds found for {fold_name}, skipping this fold.")
            return
        
        # NEW: Dictionary to store predictions for this fold
        fold_predictions = {}
        fold_hyperparams = {'fold': fold_name}  # NEW: Track hyperparameters
        
        # VERIFY FAIRNESS FIRST
        if fold_name == "fold_0":
            fairness_ok = verify_experimental_fairness(
                df_train_per_seed, df_val_per_seed, 
                build_sequences, OUTPUT_DIR
            )
            if not fairness_ok:
                print("\nCRITICAL: Data leakage detected! Fix splits before proceeding.")
                return
            
        # Baseline Single
        mean_base_single, std_base_single = baseline_single(df_val_per_seed)
        m_base_single = {k: mean_base_single.get(k, np.nan) for k in ["auprc", "precision", "recall", "f1", "tp", "tn", "fp", "fn"]}
        m_base_single.update({"fold": fold_name, "method": "baseline_single"})
        all_results.append(m_base_single)

        # Baseline Ensemble 
        df_per_seed = align_seed_frames(df_val_per_seed)
        probs = np.mean([df_per_seed[s]["p_defective"].values for s in df_per_seed], axis=0)
        preds = (probs >= 0.5).astype(int)
        df_base = list(df_per_seed.values())[0]
        m_base, pred_base = compute_metrics(df_base["y_true_defective"], preds, probs, return_predictions=True)
        m_base.update({"fold": fold_name, "method": "baseline_ensemble"})
        print_metrics("Baseline Ensemble (val)", m_base)
        all_results.append(m_base)
        fold_predictions['baseline_ensemble'] = pred_base  # NEW

        # Confidence-weighted tuned 
        print("\n→ Tuning Confidence-Weighted smoothing on train set ...")
        sigma, thr = tune_confidence_weighted(df_train_per_seed)
        fold_hyperparams['confidence_weighted'] = {'sigma': float(sigma), 'conf_thr': float(thr)}  # NEW
        
        # Get predictions
        df_per_seed = align_seed_frames(df_val_per_seed)
        probs_all = np.array([df_per_seed[s]["p_defective"].values for s in df_per_seed])
        avg_probs, std_probs = np.mean(probs_all, axis=0), np.std(probs_all, axis=0)
        df_ens = list(df_per_seed.values())[0].copy()
        df_ens["p_defective"], df_ens["p_std"] = avg_probs, std_probs
        smoothed = np.zeros(len(df_ens))
        for probs, _, idxs in build_sequences(df_ens):
            if len(probs) < 3:
                smoothed[idxs] = probs
                continue
            seq_stds = np.array([df_ens.loc[i, "p_std"] for i in idxs])
            smooth_seq = gaussian_filter1d(probs, sigma=sigma, mode="nearest")
            conf_w = 1.0 - np.clip(seq_stds / thr, 0, 1)
            smoothed[idxs] = conf_w * probs + (1 - conf_w) * smooth_seq
        preds = (smoothed >= 0.5).astype(int)
        
        m_conf, pred_conf = compute_metrics(df_ens["y_true_defective"], preds, smoothed, return_predictions=True)
        m_conf.update({"fold": fold_name, "method": "confidence_weighted_tuned", "sigma": sigma, "conf_thr": thr})
        print_metrics("Confidence-Weighted (val tuned)", m_conf)
        all_results.append(m_conf)
        fold_predictions['confidence_weighted_tuned'] = pred_conf  # NEW

        # Confidence-weighted Median tuned 
        print("\n→ Tuning Confidence-Weighted Median smoothing on train set ...")
        k_best, thr_best = tune_confidence_weighted_median(df_train_per_seed)
        fold_hyperparams['confidence_median'] = {'kernel': int(k_best), 'conf_thr': float(thr_best)}  # NEW
        
        df_per_seed = align_seed_frames(df_val_per_seed)
        probs_all = np.array([df_per_seed[s]["p_defective"].values for s in df_per_seed])
        avg_probs, std_probs = np.mean(probs_all, axis=0), np.std(probs_all, axis=0)
        df_ens = list(df_per_seed.values())[0].copy()
        df_ens["p_defective"], df_ens["p_std"] = avg_probs, std_probs
        smoothed = np.zeros(len(df_ens))
        for probs, _, idxs in build_sequences(df_ens):
            if len(probs) < 3:
                smoothed[idxs] = probs
                continue
            seq_stds = np.array([df_ens.loc[i, "p_std"] for i in idxs])
            sm = medfilt(probs, kernel_size=k_best)
            conf_w = 1.0 - np.clip(seq_stds / thr_best, 0, 1)
            smoothed[idxs] = conf_w * probs + (1 - conf_w) * sm
        preds = (smoothed >= 0.5).astype(int)
        
        m_conf_median, pred_conf_median = compute_metrics(df_ens["y_true_defective"], preds, smoothed, return_predictions=True)
        m_conf_median.update({"fold": fold_name, "method": "confidence_weighted_median_tuned", "kernel": k_best, "conf_thr": thr_best})
        print_metrics("Confidence-Weighted Median (val tuned)", m_conf_median)
        all_results.append(m_conf_median)
        fold_predictions['confidence_weighted_median_tuned'] = pred_conf_median  # NEW

        # Multiscale tuned 
        print("\n→ Tuning Multiscale Temporal smoothing on train set ...")
        sigmas_best = tune_multiscale(df_train_per_seed)
        fold_hyperparams['multiscale'] = {'sigmas': [float(s) for s in sigmas_best]}  # NEW
        
        df_per_seed = align_seed_frames(df_val_per_seed)
        probs_all = np.array([df_per_seed[s]["p_defective"].values for s in df_per_seed])
        avg_probs = np.mean(probs_all, axis=0)
        df_ens = list(df_per_seed.values())[0].copy()
        df_ens["p_defective"] = avg_probs
        multi_probs = []
        for sigma in sigmas_best:
            s_probs = np.zeros(len(df_ens))
            for probs, _, idxs in build_sequences(df_ens):
                sm = gaussian_filter1d(probs, sigma=sigma, mode="nearest") if len(probs) >= 3 else probs
                s_probs[idxs] = sm
            multi_probs.append(s_probs)
        final = np.mean(multi_probs, axis=0)
        preds = (final >= 0.5).astype(int)
        
        m_multi, pred_multi = compute_metrics(df_ens["y_true_defective"], preds, final, return_predictions=True)
        m_multi.update({"fold": fold_name, "method": "multiscale_tuned", "sigmas": str(sigmas_best)})
        print_metrics("Multiscale (val tuned)", m_multi)
        all_results.append(m_multi)
        fold_predictions['multiscale_tuned'] = pred_multi  # NEW
        
        # Defect Voting tuned 
        print("\n→ Tuning Defect Voting on train set ...")
        w_best, thr_best = tune_defect_voting(df_train_per_seed)
        fold_hyperparams['defect_voting'] = {'window': int(w_best), 'conf_thr': float(thr_best)}  # NEW
        
        df_per_seed = align_seed_frames(df_val_per_seed)
        probs_all = np.array([df_per_seed[s]["p_defective"].values for s in df_per_seed])
        avg_probs, std_probs = np.mean(probs_all, axis=0), np.std(probs_all, axis=0)
        df_ens = list(df_per_seed.values())[0].copy()
        df_ens["p_defective"], df_ens["p_std"] = avg_probs, std_probs
        voted_probs = np.zeros(len(df_ens))
        half_w = w_best // 2
        for probs, _, idxs in build_sequences(df_ens):
            seq_stds = np.array([df_ens.loc[i, "p_std"] for i in idxs])
            seq_votes = np.zeros_like(probs)
            for t in range(len(probs)):
                start = max(0, t - half_w)
                end = min(len(probs), t + half_w + 1)
                local = probs[start:end]
                local_bin = (local >= 0.5).astype(int)
                seq_votes[t] = np.mean(local_bin)
            conf_w = 1.0 - np.clip(seq_stds / thr_best, 0, 1)
            voted_probs[idxs] = conf_w * probs + (1 - conf_w) * seq_votes
        preds = (voted_probs >= 0.5).astype(int)
        
        m_vote, pred_vote = compute_metrics(df_ens["y_true_defective"], preds, voted_probs, return_predictions=True)
        m_vote.update({"fold": fold_name, "method": "defect_voting_tuned", "window": w_best, "conf_thr": thr_best})
        print_metrics("Defect Voting (val tuned)", m_vote)
        all_results.append(m_vote)
        fold_predictions['defect_voting_tuned'] = pred_vote  # NEW

        # Min Cluster Size tuned
        print("\n→ Tuning Min Cluster Size filtering on train set ...")
        size_best = tune_min_cluster_size(df_train_per_seed)
        fold_hyperparams['min_cluster'] = {'min_cluster_size': int(size_best)}  # NEW
        
        df_per_seed = align_seed_frames(df_val_per_seed)
        probs_all = np.array([df_per_seed[s]["p_defective"].values for s in df_per_seed])
        avg_probs = np.mean(probs_all, axis=0)
        df_ens = list(df_per_seed.values())[0].copy()
        df_ens["p_defective"] = avg_probs
        filtered_probs = np.zeros(len(df_ens))
        for probs, _, idxs in build_sequences(df_ens):
            preds_temp = (probs >= 0.5).astype(int)
            filtered = preds_temp.copy()
            start = 0
            for val, group in groupby(preds_temp):
                length = len(list(group))
                if val == 1 and length < size_best:
                    filtered[start:start+length] = 0
                start += length
            filtered_probs[idxs] = filtered
        preds = (filtered_probs >= 0.5).astype(int)
        
        m_cluster, pred_cluster = compute_metrics(df_ens["y_true_defective"], preds, filtered_probs, return_predictions=True)
        m_cluster.update({"fold": fold_name, "method": "min_cluster_size_tuned", "min_cluster_size": size_best})
        print_metrics("Min Cluster Size (val tuned)", m_cluster)
        all_results.append(m_cluster)
        fold_predictions['min_cluster_size_tuned'] = pred_cluster  # NEW

        # Isolated Removal tuned 
        print("\n→ Tuning Isolated Removal on train set ...")
        ctx_best = tune_isolated_removal(df_train_per_seed)
        fold_hyperparams['isolated_removal'] = {'context': int(ctx_best)}  # NEW
        
        df_per_seed = align_seed_frames(df_val_per_seed)
        probs_all = np.array([df_per_seed[s]["p_defective"].values for s in df_per_seed])
        avg_probs = np.mean(probs_all, axis=0)
        df_ens = list(df_per_seed.values())[0].copy()
        df_ens["p_defective"] = avg_probs
        cleaned_probs = np.zeros(len(df_ens))
        for probs, _, idxs in build_sequences(df_ens):
            preds_temp = (probs >= 0.5).astype(int)
            filtered = preds_temp.copy()
            for i in range(len(preds_temp)):
                if preds_temp[i] == 1:
                    start = max(0, i - ctx_best)
                    end = min(len(preds_temp), i + ctx_best + 1)
                    neighbors = np.delete(preds_temp[start:end], i - start)
                    if np.sum(neighbors) == 0:
                        filtered[i] = 0
            cleaned_probs[idxs] = filtered
        preds = (cleaned_probs >= 0.5).astype(int)
        
        m_iso, pred_iso = compute_metrics(df_ens["y_true_defective"], preds, cleaned_probs, return_predictions=True)
        m_iso.update({"fold": fold_name, "method": "isolated_removal_tuned", "context": ctx_best})
        print_metrics("Isolated Removal (val tuned)", m_iso)
        all_results.append(m_iso)
        fold_predictions['isolated_removal_tuned'] = pred_iso  # NEW

        # Temporal Smoothing tuned 
        print("\n→ Tuning Temporal Smoothing on train set ...")
        w_best = tune_temporal_smoothing(df_train_per_seed)
        fold_hyperparams['temporal_smoothing'] = {'window': int(w_best)}  # NEW
        
        df_per_seed = align_seed_frames(df_val_per_seed)
        probs_all = np.array([df_per_seed[s]["p_defective"].values for s in df_per_seed])
        avg_probs = np.mean(probs_all, axis=0)
        df_ens = list(df_per_seed.values())[0].copy()
        df_ens["p_defective"] = avg_probs
        smoothed_probs = np.zeros(len(df_ens))
        half_w = w_best // 2
        for probs, _, idxs in build_sequences(df_ens):
            smooth_seq = np.zeros_like(probs)
            for i in range(len(probs)):
                start = max(0, i - half_w)
                end = min(len(probs), i + half_w + 1)
                smooth_seq[i] = np.mean(probs[start:end])
            smoothed_probs[idxs] = smooth_seq
        preds = (smoothed_probs >= 0.5).astype(int)
        
        m_temp, pred_temp = compute_metrics(df_ens["y_true_defective"], preds, smoothed_probs, return_predictions=True)
        m_temp.update({"fold": fold_name, "method": "temporal_smoothing_tuned", "window": w_best})
        print_metrics("Temporal Smoothing (val tuned)", m_temp)
        all_results.append(m_temp)
        fold_predictions['temporal_smoothing_tuned'] = pred_temp  # NEW

        # Confidence-weighted + Isolated Removal tuned 
        print("\n→ Tuning Confidence-Weighted + Isolated Removal combination on train set ...")
        sigma, thr, ctx = tune_conf_weighted_then_isolated(df_train_per_seed)
        fold_hyperparams['conf_weighted_iso'] = {'sigma': float(sigma), 'conf_thr': float(thr), 'context': int(ctx)}  # NEW
        
        m_conf_iso = conf_weighted_then_isolated_removal(df_val_per_seed, sigma, thr, context=ctx)
        m_conf_iso.update({"fold": fold_name, "method": "conf_weighted_iso_tuned", "sigma": sigma, "conf_thr": thr, "context": ctx})
        print_metrics("Confidence-Weighted + Isolated Removal (val tuned)", m_conf_iso)
        all_results.append(m_conf_iso)
        # Note: This returns binary predictions only, so we'll skip detailed pred saving

        # Multiscale + Defect Voting tuned 
        print("\n→ Tuning Multiscale + Defect Voting combination on train set ...")
        sigmas_best, window_best, thr_best = tune_multiscale_then_voting(df_train_per_seed)
        fold_hyperparams['multiscale_vote'] = {'sigmas': [float(s) for s in sigmas_best], 'window': int(window_best), 'conf_thr': float(thr_best)}  # NEW
        
        m_multi_vote = multiscale_then_defect_voting(df_val_per_seed, sigmas_best, window=window_best, conf_thr=thr_best)
        m_multi_vote.update({"fold": fold_name, "method": "multiscale_vote_tuned", "sigmas": str(sigmas_best), "window": window_best, "conf_thr": thr_best})
        print_metrics("Multiscale + Defect Voting (val tuned)", m_multi_vote)
        all_results.append(m_multi_vote)

        # Confidence-weighted Median + Min Cluster tuned 
        print("\n→ Tuning Confidence-Weighted Median + Min Cluster combination on train set ...")
        k_best, thr_best, size_best = tune_conf_median_then_min_cluster(df_train_per_seed)
        fold_hyperparams['conf_median_cluster'] = {'kernel': int(k_best), 'conf_thr': float(thr_best), 'min_cluster_size': int(size_best)}  # NEW
        
        m_conf_median_cluster = conf_median_then_min_cluster(df_val_per_seed, kernel_size=k_best, conf_thr=thr_best, min_cluster_size=size_best)
        m_conf_median_cluster.update({"fold": fold_name, "method": "conf_median_mincluster_tuned", "kernel": k_best, "conf_thr": thr_best, "min_cluster_size": size_best})
        print_metrics("Confidence-Weighted Median + Min Cluster (val tuned)", m_conf_median_cluster)
        all_results.append(m_conf_median_cluster)

        # Confidence-weighted + Multiscale tuned 
        print("\n→ Tuning Confidence-Weighted + Multiscale combination on train set ...")
        sigma_best, thr_best, sigmas_best = tune_conf_weighted_then_multiscale(df_train_per_seed)
        fold_hyperparams['conf_weighted_multiscale'] = {'sigma': float(sigma_best), 'conf_thr': float(thr_best), 'sigmas_multiscale': [float(s) for s in sigmas_best]}  # NEW
        
        m_conf_multi = conf_weighted_then_multiscale(df_val_per_seed, sigma=sigma_best, conf_thr=thr_best, sigmas_multiscale=sigmas_best)
        m_conf_multi.update({"fold": fold_name, "method": "conf_weighted_multiscale_tuned", "sigma": sigma_best, "conf_thr": thr_best, "sigmas_multiscale": str(sigmas_best)})
        print_metrics("Confidence-Weighted + Multiscale (val tuned)", m_conf_multi)
        all_results.append(m_conf_multi)

        # LSTM TEMPORAL REFINEMENT
        print("\n→ LSTM Temporal Refinement")
        print("-" * 70)
        
        train_sequences = prepare_sequences_for_lstm(df_train_per_seed, build_sequences)
        
        print(f"  → Tuning hyperparameters (train set has {len(train_sequences)} sequences)...")
        best_lstm_params, tune_metrics = tune_lstm(train_sequences, device=device)
        fold_hyperparams['lstm'] = best_lstm_params  # NEW
        
        print("  → Training final model and evaluating on validation set...")
        m_lstm, final_model = lstm_temporal_refinement(
            df_train_per_seed,
            df_val_per_seed,
            build_sequences,
            best_lstm_params,
            device=device
        )
        
        m_lstm.update({"fold": fold_name, "method": "lstm_temporal_refinement", **best_lstm_params})
        print_metrics("LSTM Temporal Refinement (val tuned)", m_lstm)
        all_results.append(m_lstm)
        
        lstm_models[fold_name] = final_model
        val_data_per_fold[fold_name] = df_val_per_seed
        model_path = OUTPUT_DIR / f"lstm_model_{fold_name}.pt"
        torch.save({
            'model_state_dict': final_model.state_dict(),
            'params': best_lstm_params,
            'val_metrics': m_lstm,
            'fold': fold_name
        }, model_path)
        print(f"   Saved model: {model_path}")
        
        # NEW: Save fold predictions
        save_fold_predictions(df_val_per_seed, fold_predictions, fold_name, OUTPUT_DIR)
        
        # NEW: Save fold hyperparameters
        all_hyperparams.append(fold_hyperparams)
       
        with open(OUTPUT_DIR / f"hyperparams_{fold_name}.json", 'w') as f:
            json.dump(fold_hyperparams, f, indent=2)
        print(f"   Saved hyperparameters: hyperparams_{fold_name}.json")

    # After all folds processed
    print("GENERATING LSTM CROSS-FOLD VISUALIZATIONS")
    
    val_sequences_dict = {}
    for fold_name, df_val_per_seed in val_data_per_fold.items():
        val_sequences_dict[fold_name] = prepare_sequences_for_lstm(df_val_per_seed, build_sequences)
    
    visualize_lstm_fold_improvements(lstm_models, val_sequences_dict, OUTPUT_DIR, device=device)

    # Save all results
    results_df = pd.DataFrame(all_results)
    results_df.to_csv(OUTPUT_DIR / "sequence_ensemble_tuned_results.csv", index=False)
    print(f"\n Saved to: {OUTPUT_DIR / 'sequence_ensemble_tuned_results.csv'}")

    # NEW: Save all hyperparameters together

    with open(OUTPUT_DIR / "all_hyperparams.json", 'w') as f:
        json.dump(all_hyperparams, f, indent=2)
    print(f" Saved all hyperparameters: all_hyperparams.json")

    # Summary
    print("\n" + "-" * 10)
    print("MEAN RESULTS ACROSS FOLDS")
    print("-" * 10)
    metrics = ["auprc", "precision", "recall", "f1", "tp", "tn", "fp", "fn"]

    for method in results_df["method"].unique():
        data = results_df[results_df["method"] == method]
        mean = data[metrics].mean()
        std = data[metrics].std()

        print(f"\n{method.upper()}:")
        for metric in metrics:
            print(f" {metric:>10s}: {mean[metric]:.4f} ± {std[metric]:.4f}")

    # Method Rankings 
    print("DETAILED METHOD RANKINGS")
    
    prob_ranking = print_method_ranking(results_df, category='probability_based')
    binary_ranking = print_method_ranking(results_df, category='binary_only')

    # Select Best Method
    best_method_name, best_category, candidates, gains = select_best_method_by_precision(
        results_df, 
        baseline_method="baseline_ensemble",
        recall_threshold_mode='maintain'
    )
    
    # Save best method info
    best_method_info = {
        'best_method': best_method_name,
        'category': best_category,
        'selection_criterion': 'max_precision_with_recall_constraint',
        'recall_threshold_mode': 'maintain',
        'selected_date': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S'),
        **gains
    }
    
    if candidates is not None:
        best_row = candidates[candidates['method'] == best_method_name].iloc[0]
        best_method_info.update(best_row.to_dict())
        
    with open(OUTPUT_DIR / "best_method.json", "w") as f:
        json.dump(best_method_info, f, indent=2, default=str)
    print(f"\n Best method info saved to: {OUTPUT_DIR / 'best_method.json'}")
    
    # NEW: Create comprehensive results package
    results_package = {
        'metadata': {
            'experiment': 'exp4_sequence_aware_ensembles',
            'date': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S'),
            'random_seed': RANDOM_SEED,
            'device': device,
            'num_folds': len(fold_dirs),
            'exp3_root': str(EXP3_ROOT),
            'folds_root': str(FOLDS_ROOT)
        },
        'summary_metrics': results_df.to_dict('records'),
        'best_method': best_method_info,
        'method_rankings': {
            'probability_based': prob_ranking.to_dict('records') if prob_ranking is not None else [],
            'binary_only': binary_ranking.to_dict('records') if binary_ranking is not None else []
        },
        'gains_analysis': gains,
        'hyperparameters_per_fold': all_hyperparams
    }
    
    with open(OUTPUT_DIR / "complete_results_package.json", 'w') as f:
        json.dump(results_package, f, indent=2, default=str)
    print(f" Saved complete results package: complete_results_package.json")
    
    # NEW: Create plotting-ready summary
    plotting_summary = {
        'per_fold_metrics': results_df.to_dict('records'),
        'mean_metrics': results_df.groupby('method')[metrics].mean().to_dict(),
        'std_metrics': results_df.groupby('method')[metrics].std().to_dict(),
        'best_method': best_method_name,
        'baseline_metrics': results_df[results_df['method'] == 'baseline_ensemble'][metrics].mean().to_dict(),
        'hyperparameters': all_hyperparams,
        'metadata': results_package['metadata']
    }
    
    # Save as JSON
    with open(OUTPUT_DIR / "plotting_data.json", 'w') as f:
        json.dump(plotting_summary, f, indent=2, default=str)
    print(f" Saved plotting data: plotting_data.json")
    
    # Save as pickle for easy loading in Python
    
    with open(OUTPUT_DIR / "plotting_data.pkl", 'wb') as f:
        pickle.dump(plotting_summary, f)
    print(f" Saved plotting data (pickle): plotting_data.pkl")

    print("\n" + "-" * 10)
    print("EVALUATION COMPLETE!")
    print("-" * 10)
    print("\nSaved files:")
    print("  - sequence_ensemble_tuned_results.csv (main metrics table)")
    print("  - predictions/{fold}_{method}.csv (detailed predictions)")
    print("  - hyperparams_{fold}.json (per-fold hyperparameters)")
    print("  - all_hyperparams.json (all hyperparameters)")
    print("  - complete_results_package.json (everything in one file)")
    print("  - plotting_data.json/pkl (ready-to-plot summary)")
    print("  - best_method.json (best method info)")
    print("  - method_ranking_by_precision.csv (method ranking)")
    print("  - lstm_model_{fold}.pt (LSTM models)")
    print("  - lstm_comprehensive_analysis.png (LSTM visualizations)")
    print("  - lstm_fold_metrics.csv (LSTM metrics)")

if __name__ == "__main__":
    main()

# plots

In [ ]:
"""
Plotting script for Exp 4 results - 
Reads saved results and creates quality plots
"""


# Configuration
RESULTS_DIR = Path("exp4_results")
PLOTS_DIR = RESULTS_DIR / "thesis_plots"
PLOTS_DIR.mkdir(exist_ok=True)

# Set y style
plt.rcParams.update({
    'font.size': 12,
    'axes.labelsize': 12,
    'axes.titlesize': 12,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'legend.fontsize': 12,
    'figure.titlesize': 12,
    'figure.dpi': 300
})

# Load data
print("Loading results...")
results_df = pd.read_csv(RESULTS_DIR / "sequence_ensemble_tuned_results.csv")
with open(RESULTS_DIR / "plotting_data.json", 'r') as f:
    plotting_data = json.load(f)



# PLOT 2: Ensemble vs All Methods (Delta Improvements)
def plot_method_improvements():
    """Compare all methods against baseline ensemble using delta metrics - SORTED BY AUPRC."""
    print("\nCreating Plot 2: Method Improvements (Delta)...")
    
    # Exclude baseline_single, keep baseline_ensemble as reference
    methods_data = results_df[results_df['method'] != 'baseline_single'].copy()
    
    # Compute mean metrics per method
    method_means = methods_data.groupby('method')[['auprc', 'precision', 'recall', 'f1']].mean()
    
    # Get baseline ensemble metrics
    baseline = method_means.loc['baseline_ensemble']
    
    # Compute deltas
    deltas = method_means - baseline
    deltas = deltas.drop('baseline_ensemble')  # Remove baseline itself
    
    # Sort by AUPRC improvement (primary), then F1 (secondary)
    deltas = deltas.sort_values(['auprc', 'f1'], ascending=[True, True])
    
    # Create figure with 4 subplots
    fig, axes = plt.subplots(2, 2, figsize=(18, 8))
    fig.suptitle('Performance Improvements Over Baseline Ensemble (Δ Metrics)', fontsize=25, y=0.995)
    
    metrics = ['auprc', 'precision', 'recall', 'f1']
    titles = ['Δ AUPRC', 'Δ Precision', 'Δ Recall', 'Δ F1']
    
    for idx, (ax, metric, title) in enumerate(zip(axes.flat, metrics, titles)):
        # Use consistent ordering across all subplots (sorted by AUPRC)
        values = deltas[metric]
        y_pos = np.arange(len(values))
        
        # Color bars: green if positive, red if negative
        colors = ['#27ae60' if v >= 0 else '#e74c3c' for v in values]
        
        # Horizontal bar plot
        bars = ax.barh(y_pos, values, color=colors, alpha=0.85, 
                      edgecolor='black', linewidth=1.2)
        
        # Add value labels (smart positioning to avoid overlap)
        for i, (bar, val) in enumerate(zip(bars, values)):
            # Position label inside bar if bar is wide enough, otherwise outside
            if abs(val) > 0.011:  # Inside bar
                x_pos = val / 2
                ha = 'center'
                color = 'white'
            else:  # Outside bar
                x_pos = val + (0.003 if val > 0 else -0.003)
                ha = 'left' if val > 0 else 'right'
                color = 'black'
            
            ax.text(x_pos, i, f'{val:+.4f}', va='center', ha=ha, 
                   fontsize=10,color=color)
        
        # Method names (shortened to avoid overlap)
        method_labels = []
        for name in values.index:
            # Shorten method names
            label = name.replace('_tuned', '').replace('_', ' ')
            if len(label) > 35:
                label = label[:32] + '...'
            method_labels.append(label.title())
        
        # Styling
        ax.set_yticks(y_pos)
        ax.set_yticklabels(method_labels, fontsize=12)
        ax.set_xlabel(title, fontsize=12)
        ax.axvline(0, color='black', linewidth=2.5, linestyle='-', zorder=0)
        ax.grid(axis='x', alpha=0.3, linestyle='--')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        
        # Add improvement/degradation indicators on the side
        xlim = ax.get_xlim()
        ylim = ax.get_ylim()
        
        # Count improvements and degradations
        n_improvements = sum(1 for v in values if v > 0)
        n_degradations = sum(1 for v in values if v < 0)
        
        # Add summary text box
        summary_text = f"Improved: {n_improvements}\nDegraded: {n_degradations}"
        ax.text(0.02, 0.98, summary_text, transform=ax.transAxes,
        ha='left', va='top', fontsize=10, fontweight='normal',
        bbox=dict(boxstyle='round,pad=0.5', facecolor='white',
                  alpha=0.9, edgecolor='gray', linewidth=1.5))

    
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / "02_method_improvements_delta.png", dpi=300, bbox_inches='tight')
    print(f"Saved: 02_method_improvements_delta.png")
    plt.close()
    
    # Print top 3 improvements
    print("\n  Top 3 methods by AUPRC improvement:")
    top3 = deltas.sort_values('auprc', ascending=False).head(3)
    for i, (method, row) in enumerate(top3.iterrows(), 1):
        print(f"    {i}. {method}: ΔAUPRC={row['auprc']:+.4f}, ΔF1={row['f1']:+.4f}")

    
def create_baseline_table(results_df):
    # Filter only baseline methods
    df = results_df[results_df['method'].isin(['baseline_single', 'baseline_ensemble'])]

    # Compute per-method statistics (across folds)
    out = {}

    for method in ['baseline_single', 'baseline_ensemble']:
        sub = df[df['method'] == method]

        out[method] = {
            "AUPRC": f"{sub['auprc'].mean():.4f} ± {sub['auprc'].std():.4f}",
            "Precision": f"{sub['precision'].mean():.4f} ± {sub['precision'].std():.4f}",
            "Recall": f"{sub['recall'].mean():.4f} ± {sub['recall'].std():.4f}",
            "F1": f"{sub['f1'].mean():.4f} ± {sub['f1'].std():.4f}",
        }

    # Convert to DataFrame
    table = pd.DataFrame(out).T
    table.index = ["Baseline Single-Seed", "Baseline Ensemble"]

    print("\n=== BASELINE PERFORMANCE TABLE (MEAN ± STD) ===\n")
    print(table)

    # Save CSV
    table.to_csv(PLOTS_DIR / "baseline_table.csv")

    return table


def create_best_methods_table(results_df):
    """
    Creates a table with mean ± std metrics for:
      1. Method with highest mean AUPRC
      2. Method with highest mean Precision
    Excludes baseline_single.
    """
    
    # Exclude baseline_single; keep all others
    df = results_df[results_df['method'] != 'baseline_single']

    # Compute per-method means (across folds)
    method_means = df.groupby('method')[['auprc', 'precision']].mean()

    # Identify best methods
    best_auprc_method = method_means['auprc'].idxmax()
    best_precision_method = method_means['precision'].idxmax()

    best_methods = [best_auprc_method, best_precision_method]

    # Compute table with mean ± std for selected methods
    out = {}

    for method in best_methods:
        sub = df[df['method'] == method]
        out[method.replace("_", " ").title()] = {
            "AUPRC": f"{sub['auprc'].mean():.4f} ± {sub['auprc'].std():.4f}",
            "Precision": f"{sub['precision'].mean():.4f} ± {sub['precision'].std():.4f}",
            "Recall": f"{sub['recall'].mean():.4f} ± {sub['recall'].std():.4f}",
            "F1": f"{sub['f1'].mean():.4f} ± {sub['f1'].std():.4f}",
        }

    # Build DataFrame
    table = pd.DataFrame(out).T

    print("\n=== TABLE: BEST METHODS BY AUPRC + PRECISION (MEAN ± STD) ===\n")
    print(table)

    # ---------- Save CSV ----------
    table.to_csv(PLOTS_DIR / "best_methods_table.csv")

    # ---------- Save Markdown ----------
    md = "| Method | AUPRC | Precision | Recall | F1 |\n"
    md += "|--------|--------|-----------|--------|------|\n"
    for idx, row in table.iterrows():
        md += f"| {idx} | {row['AUPRC']} | {row['Precision']} | {row['Recall']} | {row['F1']} |\n"
    with open(PLOTS_DIR / "best_methods_table.md", "w") as f:
        f.write(md)

    # ---------- Save LaTeX ----------
    latex = "\\begin{tabular}{lcccc}\n\\toprule\n"
    latex += "Method & AUPRC & Precision & Recall & F1 \\\\\n\\midrule\n"
    for idx, row in table.iterrows():
        latex += f"{idx} & {row['AUPRC']} & {row['Precision']} & {row['Recall']} & {row['F1']} \\\\\n"
    latex += "\\bottomrule\n\\end{tabular}\n"

    with open(PLOTS_DIR / "best_methods_table.tex", "w") as f:
        f.write(latex)

    return table, best_auprc_method, best_precision_method


def create_delta_table(results_df):
    """
    Produce a Δ-metrics table: (method_mean - baseline_ensemble_mean)
    for AUPRC, Precision, Recall, F1.
    """

    # Exclude baseline_single completely
    df = results_df[results_df['method'] != 'baseline_single'].copy()

    # Compute mean metrics per method (across folds)
    method_means = df.groupby('method')[['auprc', 'precision', 'recall', 'f1']].mean()

    # Extract baseline ensemble row
    baseline = method_means.loc['baseline_ensemble']

    # Compute deltas for every method
    deltas = method_means.subtract(baseline)

    # Remove baseline row (its deltas are 0)
    deltas = deltas.drop('baseline_ensemble')

    # Format into “+0.0123” style strings
    pretty = deltas.applymap(lambda x: f"{x:+.4f}")

    # Rename methods nicely
    pretty.index = [name.replace("_tuned","").replace("_"," ").title()
                    for name in pretty.index]

    # Print for inspection
    print("\n=== DELTA TABLE (METHOD – BASELINE ENSEMBLE) ===\n")
    print(pretty)

    # ----- Save CSV -----
    pretty.to_csv(PLOTS_DIR / "delta_table.csv")

    # ----- Save Markdown -----
    md = "| Method | ΔAUPRC | ΔPrecision | ΔRecall | ΔF1 |\n"
    md += "|--------|---------|-------------|----------|--------|\n"
    for idx, row in pretty.iterrows():
        md += f"| {idx} | {row['auprc']} | {row['precision']} | {row['recall']} | {row['f1']} |\n"
    with open(PLOTS_DIR / "delta_table.md", "w") as f:
        f.write(md)

    # ----- Save LaTeX (booktabs) -----
    latex = "\\begin{tabular}{lcccc}\n\\toprule\n"
    latex += "Method & $\Delta$AUPRC & $\Delta$Precision & $\Delta$Recall & $\Delta$F1 \\\\\n"
    latex += "\\midrule\n"
    for idx, row in pretty.iterrows():
        latex += f"{idx} & {row['auprc']} & {row['precision']} & {row['recall']} & {row['f1']} \\\\\n"
    latex += "\\bottomrule\n\\end{tabular}\n"
    with open(PLOTS_DIR / "delta_table.tex", "w") as f:
        f.write(latex)

    return deltas, pretty


# MAIN EXECUTION
def main():
    print("-"*10)
    print("GENERATING THESIS PLOTS")
    print("-"*10)
    
    plot_method_improvements()
    baseline_table = create_baseline_table(results_df)
    print("\nGenerating best-method table...")
    best_table, best_auprc, best_prec = create_best_methods_table(results_df)

    print(f"\nBest AUPRC method: {best_auprc}")
    print(f"Best Precision method: {best_prec}")

    deltas_raw, delta_table = create_delta_table(results_df)

    
    print("\n" + "-"*10)
    print("ALL PLOTS GENERATED!")
    print("-"*10)
    print(f"\nPlots saved in: {PLOTS_DIR}")
    print("\nGenerated plots:")
    print("  1. 01_baseline_comparison.png/pdf - Justifies ensemble approach")
    print("  2. 02_method_improvements_delta.png/pdf - Delta improvements over baseline")
    print("  3. 03_absolute_performance_comparison.png/pdf - Top methods absolute scores")
    print("  4. 04_precision_recall_tradeoff.png/pdf - Trade-off visualization")
    print("  5. 05_cross_fold_consistency.png/pdf - Consistency across folds")
    print("  6. 06_method_categories.png/pdf - Category-level comparison")

if __name__ == "__main__":
    main()